In [ ]:

from pathlib import Path
import os

REPO_URL = "https://github.com/khadijasaeed6/small-model-alignment-starter.git"
REPO_DIR = Path("/kaggle/working/small-model-alignment-starter")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists.")

%cd /kaggle/working/small-model-alignment-starter

print("Current directory:", os.getcwd())
print("Files:", os.listdir("."))


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest

!python -m pip install -q -e . --no-deps

print("Dependency installation commands completed.")


In [ ]:

import torch
import transformers
import datasets
import huggingface_hub

print("===== PACKAGE VERSIONS =====")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

print("\n===== GPU CHECK =====")
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\n===== CUDA DETAILS =====")
print("PyTorch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    x = torch.randn(100, 100, device="cuda")
    print("GPU computation test:", (x @ x).shape)
    del x
    print("GPU computation successful.")


In [ ]:

from pathlib import Path

repo = Path("/kaggle/working/small-model-alignment-starter")

print("===== REPOSITORY FILES =====")
for path in sorted(repo.rglob("*")):
    if not path.is_file() or ".git" in path.parts:
        continue
    relative = path.relative_to(repo)
    if len(relative.parts) <= 3:
        print(relative)

print("\n===== README: SETUP AND TRAINING REFERENCES =====")
readme = repo / "README.md"

if readme.exists():
    lines = readme.read_text(encoding="utf-8").splitlines()
    keywords = (
        "checkpoint", "train", "kaggle", "output",
        "install", "save", "resume", "dataset"
    )

    for i, line in enumerate(lines):
        if any(word in line.lower() for word in keywords):
            start = max(0, i - 1)
            end = min(len(lines), i + 3)
            print(f"\nLines {start + 1}-{end}:")
            print("\n".join(lines[start:end]))
else:
    print("README.md not found.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

print("===== GIT STATUS =====")
!git status --short --branch

print("\n===== GIT IGNORE RULES =====")
!cat .gitignore

print("\n===== PROJECT DEPENDENCIES =====")
!cat requirements.txt

print("\n===== PROJECT CONFIGURATION =====")
!cat pyproject.toml


In [ ]:
%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path

gitignore = Path(".gitignore")

additional_rules = """
# Generated Python package metadata
*.egg-info/

# Model weights and training checkpoints
checkpoints/
*.pt
*.pth
*.safetensors
*.bin

# Local model and dataset caches
.cache/
"""

current_text = gitignore.read_text(encoding="utf-8")

for rule in additional_rules.strip().splitlines():
    if rule and rule not in current_text.splitlines():
        current_text += "\n" + rule

gitignore.write_text(current_text.rstrip() + "\n", encoding="utf-8")

print("Updated .gitignore:")
print(gitignore.read_text(encoding="utf-8"))

print("\nGit status after the update:")
!git status --short --branch

In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path

files_to_inspect = [
    "submission_adapter.py",
    "tests/test_public_adapter_smoke.py",
    "tests/toy_lm.py",
]

for filename in files_to_inspect:
    path = Path(filename)

    print("\n" + "=" * 80)
    print(f"FILE: {filename}")
    print("=" * 80)

    if path.exists():
        for line_number, line in enumerate(
            path.read_text(encoding="utf-8").splitlines(),
            start=1
        ):
            print(f"{line_number:4}: {line}")
    else:
        print("File not found.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path

adapter_code = r'''
"""Stable grading interface for the SFT and DPO implementation."""

from __future__ import annotations

import torch
import torch.nn.functional as F


def _encode_text(tokenizer, text):
    """Tokenize text without automatically adding special tokens."""
    return tokenizer.encode(text, add_special_tokens=False)


def _build_lm_batch(tokenizer, prompts, responses, max_length):
    """Build a padded batch with loss labels only on response tokens."""
    if len(prompts) != len(responses):
        raise ValueError("prompts and responses must have the same length")
    if max_length < 2:
        raise ValueError("max_length must be at least 2")
    if len(prompts) == 0:
        raise ValueError("The batch must contain at least one example")

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    all_input_ids = []
    all_labels = []

    for prompt, response in zip(prompts, responses):
        prompt_ids = _encode_text(tokenizer, prompt)
        response_ids = _encode_text(tokenizer, response)

        # Reserve a position for EOS and prioritize keeping the response.
        response_ids = response_ids[: max_length - 1]
        response_ids = response_ids + [tokenizer.eos_token_id]

        # Leave enough room for the response and EOS.
        response_ids = response_ids[:max_length]
        prompt_budget = max_length - len(response_ids)
        prompt_ids = prompt_ids[:prompt_budget]

        input_ids = prompt_ids + response_ids
        labels = ([-100] * len(prompt_ids)) + response_ids.copy()

        all_input_ids.append(input_ids)
        all_labels.append(labels)

    batch_width = max(len(row) for row in all_input_ids)

    padded_input_ids = []
    attention_masks = []
    padded_labels = []

    for input_ids, labels in zip(all_input_ids, all_labels):
        padding_length = batch_width - len(input_ids)

        padded_input_ids.append(input_ids + [pad_id] * padding_length)
        attention_masks.append([1] * len(input_ids) + [0] * padding_length)
        padded_labels.append(labels + [-100] * padding_length)

    return {
        "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(padded_labels, dtype=torch.long),
    }


def build_sft_batch(tokenizer, prompts, responses, max_length):
    """Return input_ids, attention_mask, and response-only labels."""
    return _build_lm_batch(tokenizer, prompts, responses, max_length)


def compute_response_logprobs(model, tokenizer, prompts, responses, max_length):
    """Return one summed response-token log-probability per input row."""
    batch = build_sft_batch(tokenizer, prompts, responses, max_length)

    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")

    batch = {key: value.to(device) for key, value in batch.items()}
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
    )
    logits = outputs.logits

    # At position t-1, a causal LM predicts the token at position t.
    shift_logits = logits[:, :-1, :]
    shift_labels = batch["labels"][:, 1:]

    valid_tokens = shift_labels.ne(-100)
    safe_labels = shift_labels.masked_fill(~valid_tokens, 0)

    token_log_probs = F.log_softmax(shift_logits.float(), dim=-1)
    token_log_probs = token_log_probs.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1),
    ).squeeze(-1)

    # Ignore prompt and padding positions; sum the response-token log-probs.
    token_log_probs = token_log_probs.masked_fill(~valid_tokens, 0.0)
    return token_log_probs.sum(dim=-1)


def compute_sft_loss(model, tokenizer, prompts, responses, max_length):
    """Return the mean negative summed response log-probability."""
    response_logps = compute_response_logprobs(
        model, tokenizer, prompts, responses, max_length
    )
    return -response_logps.mean()


def build_dpo_batch(
    tokenizer, prompts, chosen_responses, rejected_responses, max_length
):
    """Return separate response-masked LM batches for chosen and rejected rows."""
    if not (
        len(prompts) == len(chosen_responses) == len(rejected_responses)
    ):
        raise ValueError(
            "prompts, chosen_responses, and rejected_responses must have equal lengths"
        )

    return {
        "chosen": _build_lm_batch(
            tokenizer, prompts, chosen_responses, max_length
        ),
        "rejected": _build_lm_batch(
            tokenizer, prompts, rejected_responses, max_length
        ),
    }


def compute_dpo_loss_from_logps(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps,
    beta,
):
    """Return the scalar mean DPO objective from per-example log probabilities."""
    if beta <= 0:
        raise ValueError("beta must be positive")

    policy_log_ratio = policy_chosen_logps - policy_rejected_logps
    reference_log_ratio = reference_chosen_logps - reference_rejected_logps
    preference_logits = beta * (policy_log_ratio - reference_log_ratio)

    return -F.logsigmoid(preference_logits).mean()


def compute_dpo_loss(
    policy_model,
    reference_model,
    tokenizer,
    prompts,
    chosen_responses,
    rejected_responses,
    beta,
    max_length,
):
    """Return a scalar mean DPO loss with a frozen reference computation."""
    batch = build_dpo_batch(
        tokenizer, prompts, chosen_responses, rejected_responses, max_length
    )

    policy_chosen_logps = compute_response_logprobs(
        policy_model, tokenizer, prompts, chosen_responses, max_length
    )
    policy_rejected_logps = compute_response_logprobs(
        policy_model, tokenizer, prompts, rejected_responses, max_length
    )

    # Reference probabilities are targets for comparison, not trainable values.
    with torch.no_grad():
        reference_chosen_logps = compute_response_logprobs(
            reference_model, tokenizer, prompts, chosen_responses, max_length
        )
        reference_rejected_logps = compute_response_logprobs(
            reference_model, tokenizer, prompts, rejected_responses, max_length
        )

    return compute_dpo_loss_from_logps(
        policy_chosen_logps,
        policy_rejected_logps,
        reference_chosen_logps,
        reference_rejected_logps,
        beta,
    )
'''

Path("submission_adapter.py").write_text(adapter_code.strip() + "\n", encoding="utf-8")

print("Updated submission_adapter.py")
print("File size:", Path("submission_adapter.py").stat().st_size, "bytes")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json
import subprocess
import datetime
import torch
import transformers
import datasets

repo = Path.cwd()
backup_dir = Path("/kaggle/working/recovery_backups")
backup_dir.mkdir(parents=True, exist_ok=True)

backup_path = backup_dir / "alignment_assignment_step8.zip"
manifest_path = backup_dir / "recovery_manifest.json"

files_to_save = [
    ".gitignore",
    "README.md",
    "requirements.txt",
    "pyproject.toml",
    "submission_adapter.py",
    "tests/__init__.py",
    "tests/test_public_adapter_smoke.py",
    "tests/toy_lm.py",
    "eval_sets/hh_rlhf_eval_300_seed42.json",
    "data/persona_lora/README.md",
    "data/persona_lora/ramsay_style_preferences.json",
    "docs/assignment_manual.pdf",
]

with ZipFile(backup_path, "w", ZIP_DEFLATED) as archive:
    for relative_name in files_to_save:
        source = repo / relative_name
        if source.exists() and source.is_file():
            archive.write(source, arcname=relative_name)
        else:
            print("Not found, skipped:", relative_name)

manifest = {
    "created_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "repository": str(repo),
    "git_commit": subprocess.run(
        ["git", "rev-parse", "HEAD"],
        capture_output=True, text=True, check=True
    ).stdout.strip(),
    "adapter_smoke_tests": "7 passed",
    "python_torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "gpu_names": [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count())
    ],
    "archive_contents": [
        name for name in files_to_save
        if (repo / name).is_file()
    ],
}

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

# Verify that the archive can be opened and list its contents.
with ZipFile(backup_path, "r") as archive:
    bad_file = archive.testzip()
    print("Archive integrity:", "OK" if bad_file is None else bad_file)
    print("Files saved:", len(archive.namelist()))

print("\nArchive:", backup_path)
print("Archive size (MB):", round(backup_path.stat().st_size / (1024 * 1024), 2))
print("Manifest:", manifest_path)
print("\nReminder: these copies are still in temporary Kaggle storage.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

import importlib.metadata as metadata

packages = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
    "pytest",
]

print("===== DEPENDENCY CHECK =====")
for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n===== WORKING FILES =====")
!git status --short --branch

print("\n===== RECOVERY ARCHIVE =====")
from pathlib import Path
backup = Path("/kaggle/working/recovery_backups/alignment_assignment_step8.zip")
print("Exists:", backup.exists())
if backup.exists():
    print("Size:", round(backup.stat().st_size / 1024, 1), "KB")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path

repo = Path.cwd()

print("=" * 60)
print("1. DATASET AND PROJECT FILES")
print("=" * 60)

for folder_name in ["data", "eval_sets", "tests", "docs"]:
    folder = repo / folder_name
    print(f"\n{folder_name}/")
    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(f"  {path.relative_to(repo)} ({path.stat().st_size / 1024:.1f} KB)")
    else:
        print("  Folder not found")

print("\n" + "=" * 60)
print("2. PERSONA DATASET INSTRUCTIONS")
print("=" * 60)

persona_readme = repo / "data/persona_lora/README.md"
if persona_readme.exists():
    print(persona_readme.read_text(encoding="utf-8"))
else:
    print("Persona README not found")

print("\n" + "=" * 60)
print("3. ASSIGNMENT README")
print("=" * 60)

readme = repo / "README.md"
if readme.exists():
    print(readme.read_text(encoding="utf-8")[:12000])

print("\nInspection complete. No training was started.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

!python -m pip install -q pypdf

from pypdf import PdfReader
from pathlib import Path

pdf_path = Path("docs/assignment_manual.pdf")
reader = PdfReader(str(pdf_path))

print("Number of pages:", len(reader.pages))

manual_text = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    manual_text.append(f"\n\n===== PAGE {page_number} =====\n\n{page_text}")

full_text = "".join(manual_text)

output_path = Path("/kaggle/working/assignment_manual_extracted.txt")
output_path.write_text(full_text, encoding="utf-8")

print("Extracted text length:", len(full_text), "characters")
print("Saved extracted copy to:", output_path)
print("\n========== ASSIGNMENT MANUAL ==========\n")
print(full_text)


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import subprocess
import json
import datetime

repo = Path.cwd()
backup_dir = Path("/kaggle/working/recovery_backups")
backup_dir.mkdir(parents=True, exist_ok=True)

archive_path = backup_dir / "alignment_assignment_recovery.zip"
manifest_path = backup_dir / "alignment_assignment_recovery_manifest.json"
manual_copy = Path("/kaggle/working/assignment_manual_extracted.txt")

files_to_save = [
    ".gitignore",
    "README.md",
    "requirements.txt",
    "pyproject.toml",
    "submission_adapter.py",
    "tests/__init__.py",
    "tests/test_public_adapter_smoke.py",
    "tests/toy_lm.py",
    "eval_sets/hh_rlhf_eval_300_seed42.json",
    "data/persona_lora/README.md",
    "data/persona_lora/ramsay_style_preferences.json",
    "docs/assignment_manual.pdf",
]

saved_files = []

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for relative_name in files_to_save:
        source = repo / relative_name
        if source.is_file():
            archive.write(source, arcname=relative_name)
            saved_files.append(relative_name)

    if manual_copy.is_file():
        archive.write(manual_copy, arcname="assignment_manual_extracted.txt")
        saved_files.append("assignment_manual_extracted.txt")

    # Preserve a readable patch showing modifications to tracked files.
    diff = subprocess.run(
        ["git", "diff", "--", ".gitignore", "submission_adapter.py"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout

    archive.writestr("recovery_changes.patch", diff)
    saved_files.append("recovery_changes.patch")

manifest = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "repository": str(repo),
    "git_commit": subprocess.run(
        ["git", "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip(),
    "saved_files": saved_files,
    "public_adapter_tests": "7 passed",
    "note": (
        "Source/configuration recovery archive only. "
        "It does not contain model weights, optimizer states, or training checkpoints."
    ),
}

manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

with ZipFile(archive_path, "r") as archive:
    assert archive.testzip() is None, "Archive integrity check failed"
    print("Archive integrity: OK")
    print("Files saved:", len(archive.namelist()))

print("Archive:", archive_path)
print("Size (MB):", round(archive_path.stat().st_size / (1024 * 1024), 2))
print("Manifest:", manifest_path)


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from datasets import load_dataset

dataset_checks = [
    {
        "name": "SFT / UltraChat",
        "repo": "HuggingFaceH4/ultrachat_200k",
        "split": "train_sft",
    },
    {
        "name": "HH-RLHF preferences",
        "repo": "Anthropic/hh-rlhf",
        "split": "train",
    },
    {
        "name": "UltraFeedback preferences",
        "repo": "HuggingFaceH4/ultrafeedback_binarized",
        "split": "train_prefs",
    },
]

for item in dataset_checks:
    print("\n" + "=" * 70)
    print(item["name"])
    print("Dataset:", item["repo"])
    print("Split:", item["split"])

    try:
        stream = load_dataset(
            item["repo"],
            split=item["split"],
            streaming=True,
        )
        example = next(iter(stream))

        print("Columns:", list(example.keys()))

        for key, value in example.items():
            preview = repr(value)
            if len(preview) > 500:
                preview = preview[:500] + "..."
            print(f"\n{key}: {preview}")

        print("\nStatus: first example loaded successfully")

    except Exception as error:
        print("\nStatus: unable to inspect this dataset")
        print("Error:", type(error).__name__, str(error)[:500])

print("\nDataset inspection finished; no training was started.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from datasets import load_dataset
from itertools import islice

# 1. Inspect SFT examples
print("=" * 70)
print("SFT: UltraChat")
sft = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
    streaming=True,
)

for i, row in enumerate(islice(sft, 2), start=1):
    print(f"\nExample {i}")
    print("Prompt type:", type(row["prompt"]).__name__)
    print("Prompt preview:", repr(row["prompt"][:180]))
    print("Message count:", len(row["messages"]))
    for message in row["messages"][-3:]:
        print(
            "Role:", message.get("role"),
            "| Content:", repr(message.get("content", "")[:180])
        )

# 2. Inspect HH-RLHF preference pairs
print("\n" + "=" * 70)
print("DPO: HH-RLHF")
hh = load_dataset(
    "Anthropic/hh-rlhf",
    split="train",
    streaming=True,
)

for i, row in enumerate(islice(hh, 2), start=1):
    chosen = row["chosen"]
    rejected = row["rejected"]

    # Find the exact common prefix of both conversations.
    common_length = 0
    for a, b in zip(chosen, rejected):
        if a != b:
            break
        common_length += 1

    print(f"\nExample {i}")
    print("Chosen characters:", len(chosen))
    print("Rejected characters:", len(rejected))
    print("Common-prefix characters:", common_length)
    print("Chosen ending:", repr(chosen[-350:]))
    print("Rejected ending:", repr(rejected[-350:]))

# 3. Check preference-score ties in the first 1,000 UltraFeedback rows
print("\n" + "=" * 70)
print("DPO: UltraFeedback score check")
uf = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs",
    streaming=True,
)

checked = 0
ties = 0
missing_scores = 0

for row in islice(uf, 1000):
    checked += 1
    chosen_score = row.get("score_chosen")
    rejected_score = row.get("score_rejected")

    if chosen_score is None or rejected_score is None:
        missing_scores += 1
    elif chosen_score == rejected_score:
        ties += 1

print("Rows inspected:", checked)
print("Equal chosen/rejected scores:", ties)
print("Missing scores:", missing_scores)
print("Unequal-score rows:", checked - ties - missing_scores)

print("\nInspection complete. No model training was started.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from datasets import load_dataset

print("=" * 70)
print("1. UltraChat: prompt versus complete messages")

sft = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
    streaming=True,
)
row = next(iter(sft))

print("\nPrompt:")
print(repr(row["prompt"][:300]))

print("\nFirst six messages:")
for i, message in enumerate(row["messages"][:6]):
    print(f"\nMessage {i} | role={message.get('role')}")
    print(repr(message.get("content", "")[:250]))

first_user = next(
    (m.get("content", "") for m in row["messages"]
     if m.get("role") == "user"),
    None,
)
print("\nPrompt equals first user message:", row["prompt"] == first_user)

print("\n" + "=" * 70)
print("2. HH-RLHF: find the last shared message boundary")

hh = load_dataset(
    "Anthropic/hh-rlhf",
    split="train",
    streaming=True,
)
row = next(iter(hh))
chosen = row["chosen"]
rejected = row["rejected"]

common_length = 0
for a, b in zip(chosen, rejected):
    if a != b:
        break
    common_length += 1

common_prefix = chosen[:common_length]

print("Common prefix ends with:", repr(common_prefix[-100:]))
print("Chosen remainder:", repr(chosen[common_length:]))
print("Rejected remainder:", repr(rejected[common_length:]))

print("\n" + "=" * 70)
print("3. UltraFeedback: inspect message roles")

uf = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs",
    streaming=True,
)
row = next(iter(uf))

print("Prompt:", repr(row["prompt"][:250]))

for field in ("chosen", "rejected"):
    print(f"\n{field.upper()} messages:")
    for message in row[field][:4]:
        print(
            "Role:", message.get("role"),
            "| Content:", repr(message.get("content", "")[:250])
        )

print("\nStructure inspection finished; no training was started.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully")
print("Vocabulary size:", len(tokenizer))
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("Padding token:", repr(tokenizer.pad_token))

# Demonstrate the format Qwen expects for a user message.
example_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Explain photosynthesis simply."}],
    tokenize=False,
    add_generation_prompt=True,
)

print("\nFormatted prompt:")
print(repr(example_prompt))

# Demonstrate formatting a complete conversation.
example_conversation = [
    {"role": "user", "content": "What is photosynthesis?"},
    {"role": "assistant", "content": "It is how plants use light to make food."},
    {"role": "user", "content": "Why is sunlight needed?"},
]

print("\nMulti-turn prompt:")
print(
    tokenizer.apply_chat_template(
        example_conversation,
        tokenize=False,
        add_generation_prompt=True,
    )
)

print("\nNo model weights were loaded; no training was started.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path

source = Path("submission_adapter.py").read_text(encoding="utf-8")
lines = source.splitlines()

start = next(
    (i for i, line in enumerate(lines) if line.startswith("def _build_lm_batch")),
    None,
)

if start is None:
    print("Could not find _build_lm_batch.")
else:
    end = next(
        (
            i for i in range(start + 1, len(lines))
            if lines[i].startswith("def ")
        ),
        len(lines),
    )
    print("\n".join(
        f"{i + 1:4}: {lines[i]}"
        for i in range(start, end)
    ))


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path

path = Path("submission_adapter.py")
source = path.read_text(encoding="utf-8")

old = '''        # Reserve a position for EOS and prioritize keeping the response.
        response_ids = response_ids[: max_length - 1]
        response_ids = response_ids + [tokenizer.eos_token_id]

        # Leave enough room for the response and EOS.
        response_ids = response_ids[:max_length]
'''

new = '''        # Append EOS first, then truncate to the maximum response length.
        # If the response is too long, EOS may be removed by truncation.
        response_ids = response_ids + [tokenizer.eos_token_id]
        response_ids = response_ids[:max_length]
'''

if old not in source:
    raise RuntimeError(
        "The expected code was not found. Nothing was changed; "
        "please inspect the function again."
    )

path.write_text(source.replace(old, new, 1), encoding="utf-8")
print("Updated response truncation in submission_adapter.py")

# Rerun the public adapter tests.
import subprocess
result = subprocess.run(
    [
        "python", "-m", "pytest", "-q",
        "tests/test_public_adapter_smoke.py",
    ],
    env={
        **__import__("os").environ,
        "PYTHONPATH": ".",
    },
    text=True,
)
print("Pytest exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Tests failed. Do not proceed to training yet.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

import torch
from submission_adapter import build_sft_batch

# Reuse the Qwen tokenizer loaded earlier in this notebook.
short_batch = build_sft_batch(
    tokenizer,
    prompts=["Question: "],
    responses=["Short answer."],
    max_length=12,
)

short_ids = short_batch["input_ids"][0]
short_labels = short_batch["labels"][0]
short_mask = short_batch["attention_mask"][0]

active_labels = short_labels[short_mask.bool()]
print("Short response:")
print("Input shape:", tuple(short_batch["input_ids"].shape))
print("Last supervised token ID:", active_labels[-1].item())
print("EOS token ID:", tokenizer.eos_token_id)
print("Ends with EOS:", active_labels[-1].item() == tokenizer.eos_token_id)
print("Prompt tokens ignored:", bool((short_labels == -100).any()))

# Long response: EOS should be truncated if the response fills the limit.
long_batch = build_sft_batch(
    tokenizer,
    prompts=["Q: "],
    responses=["word " * 100],
    max_length=8,
)

long_ids = long_batch["input_ids"][0]
long_labels = long_batch["labels"][0]
long_mask = long_batch["attention_mask"][0]
active_long_labels = long_labels[long_mask.bool()]

print("\nLong response:")
print("Input length:", int(long_mask.sum().item()))
print("Maximum length:", 8)
print("Length is within limit:", int(long_mask.sum().item()) <= 8)
print("Number of supervised tokens:", int((long_labels != -100).sum().item()))
print("EOS retained:", active_long_labels[-1].item() == tokenizer.eos_token_id)

assert active_labels[-1].item() == tokenizer.eos_token_id
assert int(long_mask.sum().item()) <= 8
assert int((short_labels == -100).sum().item()) > 0

print("\nAdapter behavior checks passed.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import subprocess
import json
import datetime

repo = Path.cwd()
backup_dir = Path("/kaggle/working/recovery_backups")
backup_dir.mkdir(parents=True, exist_ok=True)

archive_path = backup_dir / "alignment_assignment_recovery_v2.zip"

files_to_save = [
    ".gitignore",
    "README.md",
    "requirements.txt",
    "pyproject.toml",
    "submission_adapter.py",
    "tests/__init__.py",
    "tests/test_public_adapter_smoke.py",
    "tests/toy_lm.py",
    "eval_sets/hh_rlhf_eval_300_seed42.json",
    "data/persona_lora/README.md",
    "data/persona_lora/ramsay_style_preferences.json",
    "docs/assignment_manual.pdf",
]

saved_files = []

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for relative_name in files_to_save:
        source = repo / relative_name
        if source.is_file():
            archive.write(source, arcname=relative_name)
            saved_files.append(relative_name)

    manual_text = Path("/kaggle/working/assignment_manual_extracted.txt")
    if manual_text.is_file():
        archive.write(
            manual_text,
            arcname="assignment_manual_extracted.txt",
        )
        saved_files.append("assignment_manual_extracted.txt")

    diff = subprocess.run(
        ["git", "diff", "--", ".gitignore", "submission_adapter.py"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    archive.writestr("recovery_changes.patch", diff)
    saved_files.append("recovery_changes.patch")

    archive.writestr(
        "backup_manifest.json",
        json.dumps(
            {
                "created_utc": datetime.datetime.now(
                    datetime.timezone.utc
                ).isoformat(),
                "saved_files": saved_files,
                "public_tests": "7 passed",
                "note": "Source and configuration only; no model checkpoints.",
            },
            indent=2,
        ),
    )

with ZipFile(archive_path, "r") as archive:
    assert archive.testzip() is None
    print("Archive integrity: OK")
    print("Files saved:", len(archive.namelist()))

print("Backup:", archive_path)
print("Size (MB):", round(archive_path.stat().st_size / (1024 * 1024), 2))


In [ ]:
from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")

required_files = [
    "submission_adapter.py",
    "tests/test_public_adapter_smoke.py",
    "docs/assignment_manual.pdf",
    "eval_sets/hh_rlhf_eval_300_seed42.json",
    "data/persona_lora/ramsay_style_preferences.json",
]

for name in required_files:
    path = project / name
    print(f"{'OK' if path.exists() else 'MISSING'}: {name}")

In [ ]:
from datasets import get_dataset_split_names, load_dataset

datasets_to_check = [
    ("SFT", "HuggingFaceH4/ultrachat_200k", "train_sft"),
    ("HH-RLHF", "Anthropic/hh-rlhf", "train"),
    ("UltraFeedback", "HuggingFaceH4/ultrafeedback_binarized", "train_prefs"),
]

for task, dataset_name, sample_split in datasets_to_check:
    print(f"\n{'=' * 60}")
    print(f"Task: {task}")
    print(f"Dataset: {dataset_name}")

    try:
        splits = get_dataset_split_names(dataset_name)
        print("Available splits:", splits)

        sample = next(iter(load_dataset(
            dataset_name,
            split=sample_split,
            streaming=True
        )))
        print("Sample columns:", list(sample.keys()))

    except Exception as e:
        print("CHECK FAILED:", type(e).__name__, str(e)[:300])

In [ ]:
from datasets import load_dataset

# 1. SFT: instruction conversations
sft = next(iter(load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
    streaming=True
)))

print("=" * 60)
print("SFT EXAMPLE")
print("Prompt:", repr(sft["prompt"][:300]))
print("Messages type:", type(sft["messages"]).__name__)
print("Messages preview:", str(sft["messages"])[:800])


# 2. HH-RLHF: preferred and rejected conversations
hh = next(iter(load_dataset(
    "Anthropic/hh-rlhf",
    split="train",
    streaming=True
)))

print("\n" + "=" * 60)
print("HH-RLHF EXAMPLE")
print("Chosen preview:", repr(hh["chosen"][-500:]))
print("Rejected preview:", repr(hh["rejected"][-500:]))


# 3. UltraFeedback: preference comparisons
uf = next(iter(load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs",
    streaming=True
)))

print("\n" + "=" * 60)
print("ULTRAFEEDBACK EXAMPLE")
print("Prompt:", repr(uf["prompt"][:300]))
print("Chosen type:", type(uf["chosen"]).__name__)
print("Chosen preview:", str(uf["chosen"])[:800])
print("Rejected preview:", str(uf["rejected"])[:800])
print("Chosen score:", uf["score_chosen"])
print("Rejected score:", uf["score_rejected"])

In [ ]:
from datasets import load_dataset

# --------------------------------------------------
# 1. Inspect SFT conversation structure
# --------------------------------------------------
print("=" * 60)
print("SFT STRUCTURE")

sft_stream = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
    streaming=True
)

for i, row in enumerate(sft_stream.take(5), start=1):
    messages = row["messages"]
    roles = [m.get("role") for m in messages]

    print(f"\nExample {i}")
    print("Number of messages:", len(messages))
    print("Roles:", roles)
    print("First message matches prompt:",
          bool(messages) and messages[0].get("content") == row["prompt"])
    print("Last message role:", roles[-1] if roles else None)


# --------------------------------------------------
# 2. Check HH-RLHF shared conversation prefixes
# --------------------------------------------------
print("\n" + "=" * 60)
print("HH-RLHF STRUCTURE")

hh_stream = load_dataset(
    "Anthropic/hh-rlhf",
    split="train",
    streaming=True
)

for i, row in enumerate(hh_stream.take(5), start=1):
    chosen = row["chosen"]
    rejected = row["rejected"]

    # Find the longest shared prefix
    common_length = 0
    limit = min(len(chosen), len(rejected))

    while (common_length < limit and
           chosen[common_length] == rejected[common_length]):
        common_length += 1

    common_prefix = chosen[:common_length]

    print(f"\nExample {i}")
    print("Chosen/rejected share prefix:", common_length > 0)
    print("Shared prefix ends with assistant marker:",
          common_prefix.endswith("\n\nAssistant: "))
    print("Chosen continuation preview:",
          repr(chosen[common_length:common_length + 100]))
    print("Rejected continuation preview:",
          repr(rejected[common_length:common_length + 100]))


# --------------------------------------------------
# 3. Check UltraFeedback score ties
# --------------------------------------------------
print("\n" + "=" * 60)
print("ULTRAFEEDBACK SCORE CHECK")

uf_stream = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs",
    streaming=True
)

total = 0
ties = 0
unequal = 0
missing = 0

for row in uf_stream.take(1000):
    total += 1
    chosen_score = row.get("score_chosen")
    rejected_score = row.get("score_rejected")

    if chosen_score is None or rejected_score is None:
        missing += 1
    elif chosen_score == rejected_score:
        ties += 1
    else:
        unequal += 1

print("Rows inspected:", total)
print("Unequal-score pairs:", unequal)
print("Tied-score pairs:", ties)
print("Missing-score pairs:", missing)

In [ ]:
from datasets import load_dataset

hh_stream = load_dataset(
    "Anthropic/hh-rlhf",
    split="train",
    streaming=True
)

rows = list(hh_stream.take(5))
row = rows[2]  # Third example

chosen = row["chosen"]
rejected = row["rejected"]

# Find the longest common prefix
i = 0
while i < min(len(chosen), len(rejected)) and chosen[i] == rejected[i]:
    i += 1

print("Common prefix length:", i)
print("\n--- Shared prefix ending ---")
print(repr(chosen[max(0, i - 250):i]))

print("\n--- Chosen around the difference ---")
print(repr(chosen[max(0, i - 100):i + 250]))

print("\n--- Rejected around the difference ---")
print(repr(rejected[max(0, i - 100):i + 250]))

print("\n--- Last assistant marker before difference ---")
marker = "\n\nAssistant: "
position = chosen.rfind(marker, 0, i)
print("Position:", position)
print("Text after marker in shared prefix:")
print(repr(chosen[position + len(marker):i] if position >= 0 else None))

In [ ]:
import torch
import transformers
import datasets
import subprocess
from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")

print("=== Repository ===")
print("Path:", project)
print("Exists:", project.exists())
if project.exists():
    result = subprocess.run(
        ["git", "-C", str(project), "remote", "-v"],
        capture_output=True,
        text=True
    )
    print(result.stdout)

print("\n=== Package Versions ===")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

print("\n=== GPU Availability ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
import json
import random
from pathlib import Path
from huggingface_hub import hf_hub_download

project = Path("/kaggle/working/small-model-alignment-starter")
output_path = project / "eval_sets" / "alpaca_eval_300_seed42_prompts.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

if output_path.exists():
    # Keep the same evaluation subset if it was already created.
    with open(output_path, "r", encoding="utf-8") as f:
        prompts = json.load(f)
    print("Reusing existing fixed evaluation subset.")

else:
    print("Downloading the official AlpacaEval JSON file...")

    source_path = hf_hub_download(
        repo_id="tatsu-lab/alpaca_eval",
        filename="alpaca_eval.json",
        repo_type="dataset"
    )

    with open(source_path, "r", encoding="utf-8") as f:
        source_data = json.load(f)

    # Support either a JSON list or a dictionary of examples.
    if isinstance(source_data, dict):
        if "instruction" in source_data:
            source_data = [source_data]
        else:
            source_data = list(source_data.values())

    if not isinstance(source_data, list):
        raise ValueError("Unexpected format in the AlpacaEval JSON file.")

    # Keep only valid rows with an instruction.
    valid_rows = [
        row for row in source_data
        if isinstance(row, dict)
        and isinstance(row.get("instruction"), str)
        and row["instruction"].strip()
    ]

    print("Valid source prompts:", len(valid_rows))

    if len(valid_rows) < 300:
        raise ValueError("Fewer than 300 valid prompts were found.")

    # Fixed seed makes selection reproducible.
    selected_indices = random.Random(42).sample(
        range(len(valid_rows)), 300
    )

    prompts = []
    for idx in selected_indices:
        row = valid_rows[idx]
        prompts.append({
            "id": f"alpaca_eval_{idx:04d}",
            "instruction": row["instruction"],
            "source_index": idx,
            "dataset": "alpaca_eval"
        })

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(prompts, f, ensure_ascii=False, indent=2)

print("\nSaved prompts:", len(prompts))
print("Unique IDs:", len({p["id"] for p in prompts}))
print("Output file:", output_path)
print("\nFirst instruction:")
print(prompts[0]["instruction"][:500])

In [ ]:
from pathlib import Path
import json
from transformers import AutoTokenizer

project = Path("/kaggle/working/small-model-alignment-starter")
model_name = "Qwen/Qwen2.5-0.5B"

prompt_file = project / "eval_sets" / "alpaca_eval_300_seed42_prompts.json"

with open(prompt_file, "r", encoding="utf-8") as f:
    prompts = json.load(f)

print("Prompt count:", len(prompts))
print("First prompt keys:", list(prompts[0].keys()))

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", len(tokenizer))
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", repr(tokenizer.pad_token))
print("Has chat template:", tokenizer.chat_template is not None)

if tokenizer.chat_template is not None:
    print("\nChat-template preview:")
    print(tokenizer.chat_template[:500])

print("\nInstruction token length:")
encoded = tokenizer(
    prompts[0]["instruction"],
    add_special_tokens=False
)
print(len(encoded["input_ids"]), "tokens")

print("\nGPU availability will be checked before model loading.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"
device = "cuda:0"

assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() >= 2, "Expected two GPUs."

print("Loading base model onto GPU 0...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
)
model = model.to(device)
model.eval()

print("Model loaded.")
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("GPU 0:", torch.cuda.get_device_name(0))

# Test only two prompts before starting the full run.
test_prompts = prompts[:2]

for i, item in enumerate(test_prompts, start=1):
    messages = [
        {"role": "user", "content": item["instruction"]}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove the input tokens; keep only the generated answer.
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    print(f"\n{'=' * 60}")
    print(f"TEST RESPONSE {i}")
    print("Instruction:", item["instruction"][:250])
    print("Generated response:")
    print(answer[:1500] if answer else "[EMPTY RESPONSE]")

In [ ]:
import torch

test_prompts = prompts[:2]

generation_settings = {
    "max_new_tokens": 192,
    "do_sample": False,          # Greedy decoding
    "repetition_penalty": 1.08,  # Mild anti-repetition control
    "no_repeat_ngram_size": 3,   # Prevent repeated 3-token sequences
    "pad_token_id": tokenizer.eos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

print("Generation settings:", generation_settings)

for i, item in enumerate(test_prompts, start=1):
    # Plain-text format; do not apply the chat template.
    formatted_prompt = (
        "### Instruction:\n"
        + item["instruction"].strip()
        + "\n\n### Response:\n"
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cuda:0")

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            **generation_settings
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"\n{'=' * 60}")
    print(f"TEST RESPONSE {i}")
    print("Instruction:", item["instruction"][:200])
    print("Generated response:")
    print(answer[:1200] if answer else "[EMPTY RESPONSE]")

In [ ]:
import json
import time
from pathlib import Path
import torch

project = Path("/kaggle/working/small-model-alignment-starter")
eval_dir = project / "eval_sets"
eval_dir.mkdir(parents=True, exist_ok=True)

prompt_file = eval_dir / "alpaca_eval_300_seed42_prompts.json"
output_file = eval_dir / "alpaca_eval_300_seed42_baseline.json"
settings_file = eval_dir / "alpaca_eval_300_seed42_baseline_settings.json"

with open(prompt_file, "r", encoding="utf-8") as f:
    prompts = json.load(f)

settings = {
    "model": "Qwen/Qwen2.5-0.5B",
    "generator": "qwen2.5-0.5b-base",
    "dataset": "alpaca_eval",
    "prompt_format": "plain_text_instruction_response",
    "decoding": "greedy",
    "do_sample": False,
    "max_new_tokens": 192,
    "repetition_penalty": 1.08,
    "no_repeat_ngram_size": 3,
    "seed": 42,
}

with open(settings_file, "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

# Resume any previously completed responses.
if output_file.exists():
    with open(output_file, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []

completed_ids = {row["id"] for row in results}
print("Prompts:", len(prompts))
print("Already completed:", len(completed_ids))

model.eval()
start_time = time.time()

for index, item in enumerate(prompts, start=1):
    if item["id"] in completed_ids:
        continue

    formatted_prompt = (
        "### Instruction:\n"
        + item["instruction"].strip()
        + "\n\n### Response:\n"
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cuda:0")

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=settings["max_new_tokens"],
            do_sample=False,
            repetition_penalty=settings["repetition_penalty"],
            no_repeat_ngram_size=settings["no_repeat_ngram_size"],
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    results.append({
        "id": item["id"],
        "instruction": item["instruction"],
        "output": answer,
        "generator": "qwen2.5-0.5b-base",
        "dataset": "alpaca_eval",
    })

    # Save after every response to minimize lost work.
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    if index % 10 == 0 or index == len(prompts):
        elapsed = time.time() - start_time
        print(
            f"Processed {index}/{len(prompts)} | "
            f"Saved {len(results)} | "
            f"Elapsed {elapsed / 60:.1f} min"
        )

print("\nGeneration finished.")
print("Responses saved:", len(results))
print("Output:", output_file)
print("Settings:", settings_file)

In [ ]:
import json
from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")
eval_dir = project / "eval_sets"

prompt_file = eval_dir / "alpaca_eval_300_seed42_prompts.json"
output_file = eval_dir / "alpaca_eval_300_seed42_baseline.json"
settings_file = eval_dir / "alpaca_eval_300_seed42_baseline_settings.json"

with open(prompt_file, "r", encoding="utf-8") as f:
    prompts = json.load(f)

with open(output_file, "r", encoding="utf-8") as f:
    results = json.load(f)

with open(settings_file, "r", encoding="utf-8") as f:
    settings = json.load(f)

required_fields = {
    "id", "instruction", "output", "generator", "dataset"
}

ids = [row.get("id") for row in results]
prompt_ids = {row["id"] for row in prompts}

missing_fields = [
    i for i, row in enumerate(results)
    if not required_fields.issubset(row.keys())
]
empty_outputs = [
    row["id"] for row in results
    if not isinstance(row.get("output"), str)
    or not row["output"].strip()
]
wrong_metadata = [
    row["id"] for row in results
    if row.get("generator") != "qwen2.5-0.5b-base"
    or row.get("dataset") != "alpaca_eval"
]

print("=== Baseline Validation ===")
print("Prompt count:", len(prompts))
print("Generated response count:", len(results))
print("Unique response IDs:", len(set(ids)))
print("Missing required fields:", len(missing_fields))
print("Empty outputs:", len(empty_outputs))
print("Incorrect metadata entries:", len(wrong_metadata))
print("Missing prompt IDs:", len(prompt_ids - set(ids)))
print("Unexpected response IDs:", len(set(ids) - prompt_ids))

print("\n=== Decoding Settings ===")
print(json.dumps(settings, indent=2))

print("\n=== Sample Saved Response ===")
print(json.dumps(results[0], ensure_ascii=False, indent=2)[:1500])

valid = (
    len(prompts) == 300
    and len(results) == 300
    and len(set(ids)) == 300
    and not missing_fields
    and not empty_outputs
    and not wrong_metadata
    and prompt_ids == set(ids)
)

print("\nFINAL RESULT:", "PASS" if valid else "CHECK REQUIRED")

In [ ]:
from pathlib import Path
import zipfile

project = Path("/kaggle/working/small-model-alignment-starter")
backup = Path("/kaggle/working/alpaca_eval_baseline_backup.zip")

files_to_save = [
    project / "eval_sets" / "alpaca_eval_300_seed42_prompts.json",
    project / "eval_sets" / "alpaca_eval_300_seed42_baseline.json",
    project / "eval_sets" / "alpaca_eval_300_seed42_baseline_settings.json",
]

with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_save:
        if not path.exists():
            raise FileNotFoundError(path)
        zf.write(path, arcname=path.name)

with zipfile.ZipFile(backup, "r") as zf:
    assert zf.testzip() is None

print("Backup created:", backup)
print("Files included:")
for name in zipfile.ZipFile(backup).namelist():
    print("-", name)
print("ZIP size (MB):", round(backup.stat().st_size / (1024 * 1024), 2))

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")

print("Repository exists:", project.exists())
print("Adapter exists:", (project / "submission_adapter.py").exists())
print("Baseline prompts exist:",
      (project / "eval_sets/alpaca_eval_300_seed42_prompts.json").exists())
print("Baseline responses exist:",
      (project / "eval_sets/alpaca_eval_300_seed42_baseline.json").exists())
print("Backup exists:",
      Path("/kaggle/working/alpaca_eval_baseline_backup.zip").exists())

In [ ]:

from datasets import load_dataset

dataset_stream = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
    streaming=True,
)

print("Dataset columns:", dataset_stream.column_names)

for i, example in enumerate(dataset_stream.take(2)):
    print(f"\n===== Example {i + 1} =====")
    print("Prompt:", str(example.get("prompt", ""))[:500])
    print("Messages:")
    for message in example.get("messages", [])[:4]:
        print(message)


In [ ]:

from datasets import load_dataset

validation_stream = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="test_sft",
    streaming=True,
)

print("Validation columns:", validation_stream.column_names)

for i, example in enumerate(validation_stream.take(2)):
    messages = example.get("messages", [])

    print(f"\n===== Validation example {i + 1} =====")
    print("Number of messages:", len(messages))

    for message in messages[:2]:
        print("Role:", message.get("role"))
        print("Content:", message.get("content", "")[:300])


In [ ]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("Chat template available:", tokenizer.chat_template is not None)

sample_messages = [
    {"role": "user", "content": "Explain photosynthesis simply."},
    {"role": "assistant", "content": "Plants use sunlight to make food."},
]

print("\n=== Complete conversation ===")
print(tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
))

print("\n=== Prompt for generating an answer ===")
print(tokenizer.apply_chat_template(
    sample_messages[:1],
    tokenize=False,
    add_generation_prompt=True,
))


In [ ]:

from datasets import load_dataset
from pathlib import Path
import json

project = Path("/kaggle/working/small-model-alignment-starter")
output_dir = project / "data" / "sft"
output_dir.mkdir(parents=True, exist_ok=True)

def prepare_sft_split(source_split, target_count, output_path):
    stream = load_dataset(
        "HuggingFaceH4/ultrachat_200k",
        split=source_split,
        streaming=True,
    )

    prepared = []
    skipped = 0

    for example in stream:
        messages = example.get("messages", [])

        first_user = next(
            (
                message.get("content", "").strip()
                for message in messages
                if message.get("role") == "user"
                and message.get("content", "").strip()
            ),
            None,
        )

        if first_user is None:
            skipped += 1
            continue

        first_assistant = None
        user_seen = False

        for message in messages:
            role = message.get("role")
            content = message.get("content", "").strip()

            if role == "user" and content == first_user:
                user_seen = True
                continue

            if user_seen and role == "assistant" and content:
                first_assistant = content
                break

        if not first_assistant:
            skipped += 1
            continue

        prepared.append({
            "id": str(example.get("prompt_id", len(prepared))),
            "prompt": first_user,
            "response": first_assistant,
            "source": "HuggingFaceH4/ultrachat_200k",
            "source_split": source_split,
        })

        if len(prepared) == target_count:
            break

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(prepared, f, ensure_ascii=False, indent=2)

    print(f"\nSource split: {source_split}")
    print("Prepared examples:", len(prepared))
    print("Skipped examples:", skipped)
    print("Saved to:", output_path)

    if len(prepared) != target_count:
        raise ValueError(
            f"Expected {target_count} examples, got {len(prepared)}"
        )

    return prepared

train_data = prepare_sft_split(
    "train_sft",
    30000,
    output_dir / "train.json",
)

val_data = prepare_sft_split(
    "test_sft",
    1000,
    output_dir / "validation.json",
)

print("\n=== First training example ===")
print("Prompt:", train_data[0]["prompt"][:300])
print("Response:", train_data[0]["response"][:500])


In [ ]:

from pathlib import Path
import json
import zipfile

project = Path("/kaggle/working/small-model-alignment-starter")
data_dir = project / "data" / "sft"

train_path = data_dir / "train.json"
val_path = data_dir / "validation.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(val_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

def check_data(rows):
    required = {"id", "prompt", "response", "source", "source_split"}
    missing_fields = sum(
        not required.issubset(row.keys()) for row in rows
    )
    empty_prompts = sum(
        not isinstance(row.get("prompt"), str)
        or not row["prompt"].strip()
        for row in rows
    )
    empty_responses = sum(
        not isinstance(row.get("response"), str)
        or not row["response"].strip()
        for row in rows
    )
    return missing_fields, empty_prompts, empty_responses

train_checks = check_data(train_data)
val_checks = check_data(val_data)

train_ids = [row["id"] for row in train_data]
val_ids = [row["id"] for row in val_data]

print("=== Dataset Validation ===")
print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Training duplicate IDs:", len(train_ids) - len(set(train_ids)))
print("Validation duplicate IDs:", len(val_ids) - len(set(val_ids)))
print("Training missing fields / empty prompts / empty responses:", train_checks)
print("Validation missing fields / empty prompts / empty responses:", val_checks)
print("Training source:", train_data[0]["source_split"])
print("Validation source:", val_data[0]["source_split"])

# Create a backup containing both prepared datasets.
backup_path = Path("/kaggle/working/sft_datasets_backup.zip")

with zipfile.ZipFile(backup_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(train_path, arcname="train.json")
    zf.write(val_path, arcname="validation.json")

with zipfile.ZipFile(backup_path, "r") as zf:
    integrity = zf.testzip()
    assert integrity is None, f"ZIP integrity error: {integrity}"

valid = (
    len(train_data) == 30000
    and len(val_data) == 1000
    and len(train_ids) == len(set(train_ids))
    and len(val_ids) == len(set(val_ids))
    and train_checks == (0, 0, 0)
    and val_checks == (0, 0, 0)
)

print("\nBackup:", backup_path)
print("Backup size (MB):", round(backup_path.stat().st_size / (1024 * 1024), 2))
print("ZIP integrity:", "OK" if integrity is None else integrity)
print("\nFINAL RESULT:", "PASS" if valid else "CHECK REQUIRED")


In [ ]:

from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")
adapter_path = project / "submission_adapter.py"

print("Adapter file exists:", adapter_path.exists())
print("\n=== Adapter function definitions ===")

source = adapter_path.read_text(encoding="utf-8")

required_functions = [
    "build_sft_batch",
    "compute_response_logprobs",
    "compute_sft_loss",
    "build_dpo_batch",
    "compute_dpo_loss_from_logps",
    "compute_dpo_loss",
]

for name in required_functions:
    found = f"def {name}(" in source
    print(f"{name}: {'FOUND' if found else 'MISSING'}")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py


In [ ]:

import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

project = Path("/kaggle/working/small-model-alignment-starter")

# Make the repository's adapter functions importable.
import sys
sys.path.insert(0, str(project))

from submission_adapter import build_sft_batch, compute_sft_loss

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_name = "Qwen/Qwen2.5-0.5B"

with open(project / "data/sft/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)

model.eval()

# Test only two examples with a short context.
test_examples = train_data[:2]
prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for row in test_examples
]
responses = [row["response"] for row in test_examples]

batch = build_sft_batch(
    tokenizer=tokenizer,
    prompts=prompts,
    responses=responses,
    max_length=128,
)

print("Device:", device)
print("Batch input shape:", tuple(batch["input_ids"].shape))
print("Batch label shape:", tuple(batch["labels"].shape))
print("Attention mask shape:", tuple(batch["attention_mask"].shape))
print("Supervised response tokens per example:",
      (batch["labels"] != -100).sum(dim=1).tolist())

# Calculate loss without gradients or parameter updates.
with torch.no_grad():
    loss = compute_sft_loss(
        model=model,
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=128,
    )

print("SFT loss:", float(loss))
print("Loss is finite:", bool(torch.isfinite(loss)))


In [ ]:

from pathlib import Path

adapter_path = Path(
    "/kaggle/working/small-model-alignment-starter/submission_adapter.py"
)
source = adapter_path.read_text(encoding="utf-8")

for function_name in [
    "def _build_lm_batch(",
    "def build_sft_batch(",
    "def compute_response_logprobs(",
    "def compute_sft_loss(",
]:
    start = source.find(function_name)

    print(f"\n{'=' * 20} {function_name} {'=' * 20}")

    if start == -1:
        print("Function not found under this exact name.")
        continue

    next_function = source.find("\ndef ", start + len(function_name))
    end = next_function if next_function != -1 else len(source)
    print(source[start:end])


In [ ]:

from pathlib import Path

adapter_path = Path(
    "/kaggle/working/small-model-alignment-starter/submission_adapter.py"
)
source = adapter_path.read_text(encoding="utf-8")

start = source.find("def _build_lm_batch(")
end = source.find("\ndef build_sft_batch(", start)

if start == -1 or end == -1:
    raise ValueError("Could not locate _build_lm_batch boundaries")

replacement = '''def _build_lm_batch(tokenizer, prompts, responses, max_length):
    """Build a padded batch with loss labels only on response tokens."""
    if len(prompts) != len(responses):
        raise ValueError("prompts and responses must have the same length")
    if max_length < 2:
        raise ValueError("max_length must be at least 2")
    if len(prompts) == 0:
        raise ValueError("The batch must contain at least one example")

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    if pad_id is None:
        pad_id = 0

    eos_id = tokenizer.eos_token_id
    if eos_id is None:
        raise ValueError("Tokenizer must define eos_token_id")

    all_input_ids = []
    all_labels = []

    # Reserve at least one token for the prompt when possible.
    min_prompt_tokens = 1

    for prompt, response in zip(prompts, responses):
        prompt_ids = _encode_text(tokenizer, prompt)
        response_ids = _encode_text(tokenizer, response)

        # Reserve one position for EOS and some room for the prompt.
        max_response_content = max_length - min_prompt_tokens - 1
        response_ids = response_ids[:max_response_content]
        response_ids = response_ids + [eos_id]

        prompt_budget = max_length - len(response_ids)
        if len(prompt_ids) > prompt_budget:
            prompt_ids = prompt_ids[-prompt_budget:] if prompt_budget > 0 else []

        input_ids = prompt_ids + response_ids
        labels = ([-100] * len(prompt_ids)) + response_ids.copy()

        all_input_ids.append(input_ids)
        all_labels.append(labels)

    batch_width = max(len(row) for row in all_input_ids)

    padded_input_ids = []
    attention_masks = []
    padded_labels = []

    for input_ids, labels in zip(all_input_ids, all_labels):
        padding_length = batch_width - len(input_ids)
        padded_input_ids.append(input_ids + [pad_id] * padding_length)
        attention_masks.append([1] * len(input_ids) + [0] * padding_length)
        padded_labels.append(labels + [-100] * padding_length)

    return {
        "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(padded_labels, dtype=torch.long),
    }
'''

source = source[:start] + replacement + source[end:]
adapter_path.write_text(source, encoding="utf-8")

print("Updated _build_lm_batch for tokenizer compatibility.")
print("Saved:", adapter_path)


In [ ]:

%cd /kaggle/working/small-model-alignment-starter
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py


In [ ]:

import torch
import json
from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")

# Re-import the updated adapter functions.
import importlib
import submission_adapter
importlib.reload(submission_adapter)

from submission_adapter import build_sft_batch, compute_sft_loss

with open(project / "data/sft/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

test_examples = train_data[:2]

prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for row in test_examples
]
responses = [row["response"] for row in test_examples]

batch = build_sft_batch(
    tokenizer=tokenizer,
    prompts=prompts,
    responses=responses,
    max_length=128,
)

supervised_tokens = (batch["labels"] != -100).sum(dim=1)
prompt_tokens = (
    (batch["labels"] == -100) & (batch["attention_mask"] == 1)
).sum(dim=1)

with torch.no_grad():
    loss = compute_sft_loss(
        model=model,
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=128,
    )

print("Batch shape:", tuple(batch["input_ids"].shape))
print("Prompt tokens per example:", prompt_tokens.tolist())
print("Supervised response tokens:", supervised_tokens.tolist())
print("SFT loss:", float(loss))
print("Loss is finite:", bool(torch.isfinite(loss)))
print("Every example has prompt tokens:", bool((prompt_tokens > 0).all()))
print("Every example has supervised tokens:", bool((supervised_tokens > 0).all()))


In [ ]:

from pathlib import Path

adapter_path = Path(
    "/kaggle/working/small-model-alignment-starter/submission_adapter.py"
)
source = adapter_path.read_text(encoding="utf-8")

start = source.find("def _build_lm_batch(")
end = source.find("\ndef build_sft_batch(", start)

if start == -1 or end == -1:
    raise ValueError("Could not locate _build_lm_batch boundaries")

replacement = '''def _build_lm_batch(tokenizer, prompts, responses, max_length):
    """Build a padded batch with response-only loss labels."""
    if len(prompts) != len(responses):
        raise ValueError("prompts and responses must have the same length")
    if max_length < 2:
        raise ValueError("max_length must be at least 2")
    if len(prompts) == 0:
        raise ValueError("The batch must contain at least one example")

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    if pad_id is None:
        pad_id = 0

    eos_id = tokenizer.eos_token_id
    if eos_id is None:
        raise ValueError("Tokenizer must define eos_token_id")

    # Keep at least 25% of the context for the prompt, up to 128 tokens.
    prompt_reserve = min(128, max(1, max_length // 4))
    max_response_content = max_length - prompt_reserve - 1

    all_input_ids = []
    all_labels = []

    for prompt, response in zip(prompts, responses):
        prompt_ids = _encode_text(tokenizer, prompt)
        response_ids = _encode_text(tokenizer, response)

        # Truncate long responses to preserve prompt context.
        response_ids = response_ids[:max_response_content]
        response_ids = response_ids + [eos_id]

        prompt_budget = max_length - len(response_ids)
        prompt_ids = prompt_ids[-prompt_budget:] if prompt_budget > 0 else []

        input_ids = prompt_ids + response_ids
        labels = ([-100] * len(prompt_ids)) + response_ids.copy()

        all_input_ids.append(input_ids)
        all_labels.append(labels)

    batch_width = max(len(row) for row in all_input_ids)

    padded_input_ids = []
    attention_masks = []
    padded_labels = []

    for input_ids, labels in zip(all_input_ids, all_labels):
        padding_length = batch_width - len(input_ids)
        padded_input_ids.append(input_ids + [pad_id] * padding_length)
        attention_masks.append([1] * len(input_ids) + [0] * padding_length)
        padded_labels.append(labels + [-100] * padding_length)

    return {
        "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(padded_labels, dtype=torch.long),
    }
'''

source = source[:start] + replacement + source[end:]
adapter_path.write_text(source, encoding="utf-8")
print("Updated prompt/response truncation rule.")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py


In [ ]:

from pathlib import Path

adapter_path = Path(
    "/kaggle/working/small-model-alignment-starter/submission_adapter.py"
)
source = adapter_path.read_text(encoding="utf-8")

old = '''    eos_id = tokenizer.eos_token_id
    if eos_id is None:
        raise ValueError("Tokenizer must define eos_token_id")
'''

new = '''    # Qwen chat turns normally end with <|im_end|>.
    # Fall back to eos_token_id for simpler tokenizers, including test tokenizers.
    eos_id = tokenizer.eos_token_id

    if hasattr(tokenizer, "convert_tokens_to_ids"):
        chat_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        unknown_id = getattr(tokenizer, "unk_token_id", None)

        if chat_end_id is not None and chat_end_id != unknown_id:
            eos_id = chat_end_id

    if eos_id is None:
        raise ValueError("Tokenizer must define eos_token_id or <|im_end|>")
'''

count = source.count(old)
if count != 1:
    raise ValueError(
        f"Expected to find the EOS-token block exactly once; found {count}. "
        "No changes were saved."
    )

source = source.replace(old, new, 1)
adapter_path.write_text(source, encoding="utf-8")

print("Updated Qwen chat-turn ending token handling.")
print("Saved:", adapter_path)


In [ ]:

%cd /kaggle/working/small-model-alignment-starter
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py


In [ ]:

import json
import torch
import importlib
from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")

import submission_adapter
importlib.reload(submission_adapter)

from submission_adapter import build_sft_batch, compute_sft_loss

with open(project / "data/sft/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

examples = train_data[:2]

prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for row in examples
]
responses = [row["response"] for row in examples]

batch = build_sft_batch(
    tokenizer, prompts, responses, max_length=128
)

supervised = (batch["labels"] != -100).sum(dim=1)
prompt_tokens = (
    (batch["labels"] == -100) & (batch["attention_mask"] == 1)
).sum(dim=1)

with torch.no_grad():
    loss = compute_sft_loss(
        model, tokenizer, prompts, responses, max_length=128
    )

print("Input shape:", tuple(batch["input_ids"].shape))
print("Prompt tokens:", prompt_tokens.tolist())
print("Supervised tokens (including turn-ending token):", supervised.tolist())
print("Loss:", float(loss))
print("Finite loss:", bool(torch.isfinite(loss)))
print("Each example has prompt and response tokens:",
      bool(((prompt_tokens > 0) & (supervised > 0)).all()))

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
print("Qwen <|im_end|> token ID:", im_end_id)
print("Batch contains <|im_end|> as a supervised target:",
      bool(((batch["labels"] == im_end_id) & (batch["labels"] != -100)).any()))


In [ ]:

from pathlib import Path
import zipfile

project = Path("/kaggle/working/small-model-alignment-starter")
backup_dir = Path("/kaggle/working/recovery_backups")
backup_dir.mkdir(parents=True, exist_ok=True)

backup_path = backup_dir / "alignment_pre_sft_backup.zip"

files_to_backup = [
    project / "submission_adapter.py",
    project / "data/sft/train.json",
    project / "data/sft/validation.json",
    project / "eval_sets/alpaca_eval_300_seed42_prompts.json",
    project / "eval_sets/alpaca_eval_300_seed42_baseline.json",
    project / "eval_sets/alpaca_eval_300_seed42_baseline_settings.json",
]

missing = [str(p) for p in files_to_backup if not p.exists()]
if missing:
    raise FileNotFoundError("Missing files:\n" + "\n".join(missing))

with zipfile.ZipFile(backup_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_backup:
        zf.write(path, arcname=str(path.relative_to(project)))

with zipfile.ZipFile(backup_path, "r") as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Backup ZIP integrity check failed: {bad_member}")

print("Backup created:", backup_path)
print("Size (MB):", round(backup_path.stat().st_size / (1024 * 1024), 2))
print("Files backed up:", len(files_to_backup))
print("ZIP integrity: PASS")


In [ ]:

from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")

print("Top-level project files:")
for path in sorted(project.iterdir()):
    print(("DIR  " if path.is_dir() else "FILE "), path.name)

print("\nPython files:")
for path in sorted(project.rglob("*.py")):
    if ".git" not in path.parts and "__pycache__" not in path.parts:
        print(path.relative_to(project))

In [ ]:

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}: {props.name}")
    print(f"Total memory: {props.total_memory / (1024**3):.2f} GiB")
    print(f"Allocated: {torch.cuda.memory_allocated(i) / (1024**3):.2f} GiB")
    print(f"Reserved: {torch.cuda.memory_reserved(i) / (1024**3):.2f} GiB")

print("\nExisting model variable:", "model" in globals())
print("Existing tokenizer variable:", "tokenizer" in globals())


In [ ]:

from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")
script_path = project / "train_sft.py"

script = r'''
import json
import math
import os
import random
import time
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

from submission_adapter import build_sft_batch, compute_sft_loss


PROJECT = Path(__file__).resolve().parent
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
OUTPUT_DIR = PROJECT / "checkpoints" / "sft"
TRAIN_PATH = PROJECT / "data" / "sft" / "train.json"
VAL_PATH = PROJECT / "data" / "sft" / "validation.json"

SEED = int(os.environ.get("SFT_SEED", "42"))
MAX_LENGTH = int(os.environ.get("SFT_MAX_LENGTH", "512"))
BATCH_SIZE = int(os.environ.get("SFT_BATCH_SIZE", "2"))
GRAD_ACCUM = int(os.environ.get("SFT_GRAD_ACCUM", "16"))
EPOCHS = int(os.environ.get("SFT_EPOCHS", "1"))
LR = float(os.environ.get("SFT_LR", "2e-5"))
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.03
VAL_BATCHES = int(os.environ.get("SFT_VAL_BATCHES", "0"))
SAVE_EVERY = int(os.environ.get("SFT_SAVE_EVERY", "100"))
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_prompt(tokenizer, prompt):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, step,
                    best_val_loss, best_step):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(".tmp")
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "step": step,
            "best_val_loss": best_val_loss,
            "best_step": best_step,
            "seed": SEED,
        },
        temp_path,
    )
    os.replace(temp_path, path)


def main():
    set_seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    train_rows = read_json(TRAIN_PATH)
    val_rows = read_json(VAL_PATH)

    if not train_rows or not val_rows:
        raise ValueError("Training and validation datasets must not be empty.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
    ).to(DEVICE)
    model.config.use_cache = False
    model.train()

    optimizer = AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    updates_per_epoch = math.ceil(len(train_rows) / (BATCH_SIZE * GRAD_ACCUM))
    total_steps = updates_per_epoch * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # FP16 autocast is used for model computation; trainable parameters stay FP32.
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    start_epoch = 0
    global_step = 0
    best_val_loss = float("inf")
    best_step = 0
    resume_path = OUTPUT_DIR / "last_checkpoint.pt"

    if resume_path.exists() and os.environ.get("SFT_RESUME", "1") == "1":
        checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        start_epoch = checkpoint["epoch"]
        global_step = checkpoint["step"]
        best_val_loss = checkpoint["best_val_loss"]
        best_step = checkpoint["best_step"]
        print(f"Resumed from step {global_step}, epoch {start_epoch}.")

    print(f"Device: {DEVICE}")
    print(f"Training examples: {len(train_rows)}")
    print(f"Validation examples: {len(val_rows)}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Gradient accumulation: {GRAD_ACCUM}")
    print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
    print(f"Context length: {MAX_LENGTH}")
    print(f"Planned optimizer steps: {total_steps}")

    optimizer.zero_grad(set_to_none=True)
    micro_step = 0
    running_loss = 0.0
    start_time = time.time()

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        indices = list(range(len(train_rows)))
        random.shuffle(indices)

        batch_count = math.ceil(len(indices) / BATCH_SIZE)
        for batch_index, offset in enumerate(range(0, len(indices), BATCH_SIZE)):
            rows = [train_rows[i] for i in indices[offset:offset + BATCH_SIZE]]
            prompts = [make_prompt(tokenizer, row["prompt"]) for row in rows]
            responses = [row["response"] for row in rows]

            batch = build_sft_batch(
                tokenizer, prompts, responses, max_length=MAX_LENGTH
            )
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda"),
            ):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    use_cache=False,
                )
                logits = outputs.logits[:, :-1, :].contiguous()
                labels = batch["labels"][:, 1:].contiguous()
                valid = labels.ne(-100).sum()

                if valid.item() == 0:
                    raise ValueError("Batch has no response tokens for loss.")

                loss_sum = torch.nn.functional.cross_entropy(
                    logits.reshape(-1, logits.size(-1)).float(),
                    labels.reshape(-1),
                    ignore_index=-100,
                    reduction="sum",
                )
                loss = loss_sum / valid
                scaled_loss = loss / GRAD_ACCUM

            scaler.scale(scaled_loss).backward()
            running_loss += float(loss.detach())
            micro_step += 1

            is_last_batch = batch_index == batch_count - 1
            if micro_step % GRAD_ACCUM == 0 or is_last_batch:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1
                micro_step = 0

                if global_step % 10 == 0:
                    print(
                        f"epoch={epoch + 1} step={global_step}/{total_steps} "
                        f"mean_microbatch_loss={running_loss / 10:.4f} "
                        f"lr={scheduler.get_last_lr()[0]:.2e} "
                        f"elapsed_min={(time.time() - start_time) / 60:.1f}",
                        flush=True,
                    )
                    running_loss = 0.0

                if global_step % SAVE_EVERY == 0:
                    save_checkpoint(
                        resume_path, model, optimizer, scheduler, scaler,
                        epoch, global_step, best_val_loss, best_step,
                    )
                    print(f"Saved recovery checkpoint at step {global_step}.")

        # Validation uses the same response-only token-average cross-entropy.
        model.eval()
        total_nll = 0.0
        total_tokens = 0
        with torch.no_grad():
            val_indices = list(range(len(val_rows)))
            if VAL_BATCHES > 0:
                val_indices = val_indices[:VAL_BATCHES * BATCH_SIZE]

            for offset in range(0, len(val_indices), BATCH_SIZE):
                rows = [val_rows[i] for i in val_indices[offset:offset + BATCH_SIZE]]
                prompts = [make_prompt(tokenizer, row["prompt"]) for row in rows]
                responses = [row["response"] for row in rows]
                batch = build_sft_batch(
                    tokenizer, prompts, responses, max_length=MAX_LENGTH
                )
                batch = {k: v.to(DEVICE) for k, v in batch.items()}

                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=(DEVICE.type == "cuda"),
                ):
                    outputs = model(
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        use_cache=False,
                    )
                    logits = outputs.logits[:, :-1, :].contiguous().float()
                    labels = batch["labels"][:, 1:].contiguous()
                    n_tokens = labels.ne(-100).sum()
                    nll = torch.nn.functional.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        labels.reshape(-1),
                        ignore_index=-100,
                        reduction="sum",
                    )
                total_nll += float(nll)
                total_tokens += int(n_tokens)

        val_loss = total_nll / max(total_tokens, 1)
        print(
            f"Validation epoch={epoch + 1}: response_token_loss={val_loss:.4f}",
            flush=True,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_step = global_step
            save_checkpoint(
                OUTPUT_DIR / "best_checkpoint.pt",
                model, optimizer, scheduler, scaler,
                epoch + 1, global_step, best_val_loss, best_step,
            )
            print("Saved new best checkpoint.")

        save_checkpoint(
            resume_path, model, optimizer, scheduler, scaler,
            epoch + 1, global_step, best_val_loss, best_step,
        )
        save_checkpoint(
            OUTPUT_DIR / "final_checkpoint.pt",
            model, optimizer, scheduler, scaler,
            epoch + 1, global_step, best_val_loss, best_step,
        )

    print(f"Best validation loss: {best_val_loss:.4f} at step {best_step}")
    print(f"Checkpoints saved in: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()
'''

script_path.write_text(script.lstrip(), encoding="utf-8")
print("Created:", script_path)
print("Size (KB):", round(script_path.stat().st_size / 1024, 1))


In [ ]:

from pathlib import Path

project = Path("/kaggle/working/small-model-alignment-starter")
script_path = project / "train_sft.py"
source = script_path.read_text(encoding="utf-8")

replacements = [
    (
        'def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, step,\n                    best_val_loss, best_step):',
        'def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, step,\n                    best_val_loss, best_step, batch_offset=0):'
    ),
    (
        '            "best_step": best_step,\n            "seed": SEED,',
        '            "best_step": best_step,\n            "batch_offset": batch_offset,\n            "seed": SEED,'
    ),
    (
        '    train_rows = read_json(TRAIN_PATH)\n    val_rows = read_json(VAL_PATH)\n\n    if not train_rows or not val_rows:',
        '    train_rows = read_json(TRAIN_PATH)\n    val_rows = read_json(VAL_PATH)\n\n'
        '    # Optional subset for quick smoke tests.\n'
        '    max_train_examples = int(os.environ.get("SFT_MAX_TRAIN_EXAMPLES", "0"))\n'
        '    if max_train_examples > 0:\n'
        '        train_rows = train_rows[:max_train_examples]\n\n'
        '    if not train_rows or not val_rows:'
    ),
    (
        '    best_val_loss = float("inf")\n    best_step = 0\n    resume_path = OUTPUT_DIR / "last_checkpoint.pt"',
        '    best_val_loss = float("inf")\n    best_step = 0\n'
        '    resume_batch_offset = 0\n'
        '    resume_path = OUTPUT_DIR / "last_checkpoint.pt"'
    ),
    (
        '        best_val_loss = checkpoint["best_val_loss"]\n'
        '        best_step = checkpoint["best_step"]\n'
        '        print(f"Resumed from step {global_step}, epoch {start_epoch}.")',
        '        best_val_loss = checkpoint["best_val_loss"]\n'
        '        best_step = checkpoint["best_step"]\n'
        '        resume_batch_offset = checkpoint.get("batch_offset", 0)\n'
        '        print(f"Resumed from step {global_step}, epoch {start_epoch}, "\n'
        '              f"batch offset {resume_batch_offset}.")'
    ),
    (
        '        indices = list(range(len(train_rows)))\n        random.shuffle(indices)',
        '        indices = list(range(len(train_rows)))\n'
        '        random.Random(SEED + epoch).shuffle(indices)'
    ),
    (
        '        for batch_index, offset in enumerate(range(0, len(indices), BATCH_SIZE)):\n'
        '            rows = [train_rows[i] for i in indices[offset:offset + BATCH_SIZE]]',
        '        for batch_index, offset in enumerate(range(0, len(indices), BATCH_SIZE)):\n'
        '            if epoch == start_epoch and batch_index < resume_batch_offset:\n'
        '                continue\n'
        '            rows = [train_rows[i] for i in indices[offset:offset + BATCH_SIZE]]'
    ),
    (
        '                scaler.unscale_(optimizer)\n'
        '                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)',
        '                scaler.unscale_(optimizer)\n'
        '                # Correct gradient scale for an incomplete final accumulation group.\n'
        '                if is_last_batch and micro_step < GRAD_ACCUM:\n'
        '                    correction = GRAD_ACCUM / micro_step\n'
        '                    for parameter in model.parameters():\n'
        '                        if parameter.grad is not None:\n'
        '                            parameter.grad.mul_(correction)\n'
        '                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)'
    ),
    (
        '                        epoch, global_step, best_val_loss, best_step,\n'
        '                    )\n'
        '                    print(f"Saved recovery checkpoint at step {global_step}.")',
        '                        epoch, global_step, best_val_loss, best_step,\n'
        '                        batch_offset=batch_index + 1,\n'
        '                    )\n'
        '                    print(f"Saved recovery checkpoint at step {global_step}.")'
    ),
]

for old, new in replacements:
    count = source.count(old)
    if count != 1:
        raise ValueError(
            f"Expected exactly one match, found {count}. "
            "No changes have been saved."
        )
    source = source.replace(old, new, 1)

compile(source, str(script_path), "exec")
script_path.write_text(source, encoding="utf-8")

print("Training script patched successfully.")
print("Python syntax: PASS")
print("Smoke-test subset option: added")
print("Checkpoint batch-position tracking: added")
print("Incomplete gradient accumulation correction: added")


In [ ]:

from pathlib import Path

script_path = Path(
    "/kaggle/working/small-model-alignment-starter/train_sft.py"
)
source = script_path.read_text(encoding="utf-8")

old_scaler = (
    'scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))'
)
new_scaler = (
    'scaler = torch.amp.GradScaler(\n'
    '        "cuda", enabled=(DEVICE.type == "cuda"), init_scale=1024.0\n'
    '    )'
)

old_step = '''                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1
                micro_step = 0
'''

new_step = '''                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                new_scale = scaler.get_scale()

                # A decreased scale means GradScaler skipped the optimizer update.
                update_skipped = scaler.is_enabled() and new_scale < old_scale

                optimizer.zero_grad(set_to_none=True)
                micro_step = 0

                if update_skipped:
                    print(
                        "Warning: FP16 overflow detected; optimizer update skipped. "
                        f"Gradient scale reduced to {new_scale}.",
                        flush=True,
                    )
                else:
                    scheduler.step()
                    global_step += 1
'''

if source.count(old_scaler) != 1:
    raise ValueError("Could not uniquely locate the GradScaler initialization.")
if source.count(old_step) != 1:
    raise ValueError("Could not uniquely locate the optimizer update block.")

source = source.replace(old_scaler, new_scaler, 1)
source = source.replace(old_step, new_step, 1)

compile(source, str(script_path), "exec")
script_path.write_text(source, encoding="utf-8")

print("FP16 overflow handling updated.")
print("Python syntax: PASS")


In [ ]:

%cd /kaggle/working/small-model-alignment-starter

import os
import subprocess
import sys

env = os.environ.copy()
env.update({
    "SFT_MAX_TRAIN_EXAMPLES": "8",
    "SFT_MAX_LENGTH": "128",
    "SFT_BATCH_SIZE": "2",
    "SFT_GRAD_ACCUM": "2",
    "SFT_EPOCHS": "1",
    "SFT_VAL_BATCHES": "2",
    "SFT_SAVE_EVERY": "1",
    "SFT_RESUME": "0",
})

result = subprocess.run(
    [sys.executable, "-u", "train_sft.py"],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=900,
)

print(result.stdout[-12000:])
print("\nSmoke-test process exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Smoke test failed. Do not start full training yet.")


In [ ]:

from pathlib import Path
import shutil
import subprocess

project = Path("/kaggle/working/small-model-alignment-starter")
checkpoint_dir = project / "checkpoints" / "sft"

print("=== Disk space ===")
for location in ["/kaggle/working", "/tmp"]:
    usage = shutil.disk_usage(location)
    print(f"\n{location}")
    print(f"Total:     {usage.total / (1024**3):.2f} GiB")
    print(f"Used:      {usage.used / (1024**3):.2f} GiB")
    print(f"Free:      {usage.free / (1024**3):.2f} GiB")

print("\n=== Filesystem details ===")
subprocess.run(["df", "-h", "/kaggle/working"], check=False)

print("\n=== SFT checkpoint files ===")
if checkpoint_dir.exists():
    for path in sorted(checkpoint_dir.iterdir()):
        if path.is_file():
            print(f"{path.name}: {path.stat().st_size / (1024**3):.3f} GiB")
else:
    print("Checkpoint directory does not exist.")

print("\n=== Temporary checkpoint files ===")
for path in project.rglob("*.tmp"):
    if path.is_file():
        print(f"{path}: {path.stat().st_size / (1024**3):.3f} GiB")


In [ ]:

from pathlib import Path
import shutil

checkpoint_dir = Path(
    "/kaggle/working/small-model-alignment-starter/checkpoints/sft"
)

print("Files scheduled for removal:")
if checkpoint_dir.exists():
    for path in sorted(checkpoint_dir.iterdir()):
        if path.is_file():
            print(f"  {path.name} — {path.stat().st_size / (1024**3):.2f} GiB")

    # These are disposable artifacts from the failed 8-example smoke test.
    shutil.rmtree(checkpoint_dir)
    print("\nRemoved the smoke-test checkpoint directory.")
else:
    print("Directory already absent.")

usage = shutil.disk_usage("/kaggle/working")
print(f"Free disk space: {usage.free / (1024**3):.2f} GiB")


In [ ]:

from pathlib import Path

script_path = Path(
    "/kaggle/working/small-model-alignment-starter/train_sft.py"
)
lines = script_path.read_text(encoding="utf-8").splitlines()

print("=== Checkpoint saving and configuration ===")
for start, end in [(1, 100), (100, 240), (240, 360)]:
    if start <= len(lines):
        print(f"\n--- Lines {start}–{min(end, len(lines))} ---")
        for i in range(start - 1, min(end, len(lines))):
            print(f"{i + 1:3}: {lines[i]}")


In [ ]:

from pathlib import Path

script_path = Path(
    "/kaggle/working/small-model-alignment-starter/train_sft.py"
)
source = script_path.read_text(encoding="utf-8")

# 1. Put outputs in the stable run directory required by the assignment.
old_output = 'OUTPUT_DIR = PROJECT / "checkpoints" / "sft"'
new_output = 'OUTPUT_DIR = PROJECT / "outputs" / "sft_run"'
assert source.count(old_output) == 1
source = source.replace(old_output, new_output)

# 2. Replace the old full-size checkpoint function with disk-efficient helpers.
start_marker = "def save_checkpoint("
end_marker = "\n\ndef main():"
start = source.index(start_marker)
end = source.index(end_marker, start)

new_functions = r'''
def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, step,
                    best_val_loss, best_step, batch_offset=0):
    """Save one resumable checkpoint with compressed AdamW moment tensors."""
    if os.environ.get("SFT_DISABLE_CHECKPOINTS", "0") == "1":
        return

    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(".tmp")

    # Store model weights on CPU.
    model_state = {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    }

    # AdamW's two large moment buffers are stored as FP16 to save disk.
    # They are restored to FP32 after loading the checkpoint.
    optimizer_state = optimizer.state_dict()
    compressed_states = {}

    for parameter_id, state in optimizer_state["state"].items():
        compressed_states[parameter_id] = {}
        for key, value in state.items():
            if torch.is_tensor(value):
                value = value.detach().cpu()
                if key in ("exp_avg", "exp_avg_sq", "max_exp_avg_sq"):
                    value = value.to(torch.float16)
            compressed_states[parameter_id][key] = value

    optimizer_state["state"] = compressed_states

    checkpoint = {
        "model": model_state,
        "optimizer": optimizer_state,
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "step": step,
        "best_val_loss": best_val_loss,
        "best_step": best_step,
        "batch_offset": batch_offset,
        "seed": SEED,
    }

    try:
        torch.save(checkpoint, temp_path)
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def save_model_only(model, directory):
    """Save model weights/config only; no optimizer state."""
    if os.environ.get("SFT_DISABLE_CHECKPOINTS", "0") == "1":
        return

    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(
        directory,
        safe_serialization=True,
    )


def append_metrics(record):
    """Append one JSON record so completed evaluations survive interruptions."""
    metrics_path = OUTPUT_DIR / "metrics.jsonl"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    with metrics_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
'''

source = source[:start] + new_functions + source[end:]

# 3. Restore compressed optimizer moments as FP32 when resuming.
old_load = '''        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])'''
new_load = '''        optimizer.load_state_dict(checkpoint["optimizer"])
        for state in optimizer.state.values():
            for key in ("exp_avg", "exp_avg_sq", "max_exp_avg_sq"):
                value = state.get(key)
                if torch.is_tensor(value) and value.is_floating_point():
                    state[key] = value.float()
        scheduler.load_state_dict(checkpoint["scheduler"])'''
assert source.count(old_load) == 1
source = source.replace(old_load, new_load)

# 4. Make periodic saves conditional and keep the last checkpoint rolling.
old_periodic = '''                if global_step % SAVE_EVERY == 0:
                    save_checkpoint(
                        resume_path, model, optimizer, scheduler, scaler,
                        epoch, global_step, best_val_loss, best_step,
                        batch_offset=batch_index + 1,
                    )
                    print(f"Saved recovery checkpoint at step {global_step}.")'''
new_periodic = '''                if SAVE_EVERY > 0 and global_step > 0 and global_step % SAVE_EVERY == 0:
                    save_checkpoint(
                        resume_path, model, optimizer, scheduler, scaler,
                        epoch, global_step, best_val_loss, best_step,
                        batch_offset=batch_index + 1,
                    )
                    print(f"Saved rolling recovery checkpoint at step {global_step}.")'''
assert source.count(old_periodic) == 1
source = source.replace(old_periodic, new_periodic)

# 5. Replace best/final checkpoint saves with model-only saves.
old_end = '''        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_step = global_step
            save_checkpoint(
                OUTPUT_DIR / "best_checkpoint.pt",
                model, optimizer, scheduler, scaler,
                epoch + 1, global_step, best_val_loss, best_step,
            )
            print("Saved new best checkpoint.")

        save_checkpoint(
            resume_path, model, optimizer, scheduler, scaler,
            epoch + 1, global_step, best_val_loss, best_step,
        )
        save_checkpoint(
            OUTPUT_DIR / "final_checkpoint.pt",
            model, optimizer, scheduler, scaler,
            epoch + 1, global_step, best_val_loss, best_step,
        )'''
new_end = '''        append_metrics({
            "epoch": epoch + 1,
            "step": global_step,
            "validation_response_token_loss": val_loss,
            "best_validation_loss_before_update": best_val_loss,
            "elapsed_minutes": (time.time() - start_time) / 60.0,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_step = global_step
            save_model_only(model, OUTPUT_DIR / "best_checkpoint")
            print("Saved new best model weights.")

        # Keep a single rolling recovery checkpoint for resuming training.
        save_checkpoint(
            resume_path, model, optimizer, scheduler, scaler,
            epoch + 1, global_step, best_val_loss, best_step,
            batch_offset=0,
        )

        # Final model weights are saved separately from optimizer state.
        save_model_only(model, OUTPUT_DIR / "final_checkpoint")'''
assert source.count(old_end) == 1
source = source.replace(old_end, new_end)

script_path.write_text(source, encoding="utf-8")

print("Patch applied successfully.")
print("Script:", script_path)
print("Output directory: outputs/sft_run/")
print("Recovery: one rolling last_checkpoint.pt")
print("Best/final: model weights only")
print("Evaluation metrics: metrics.jsonl")


In [ ]:

from pathlib import Path
import ast

script_path = Path(
    "/kaggle/working/small-model-alignment-starter/train_sft.py"
)
source = script_path.read_text(encoding="utf-8")

# Syntax check: this does not execute the training script.
ast.parse(source)

checks = {
    "Stable outputs directory": 'PROJECT / "outputs" / "sft_run"' in source,
    "Compressed recovery optimizer state": 'value.to(torch.float16)' in source,
    "Optimizer moments restored to FP32": 'state[key] = value.float()' in source,
    "Best model saved separately": 'OUTPUT_DIR / "best_checkpoint"' in source,
    "Final model saved separately": 'OUTPUT_DIR / "final_checkpoint"' in source,
    "Evaluation metrics appended": 'def append_metrics(record):' in source,
    "Smoke-test checkpoint switch": 'SFT_DISABLE_CHECKPOINTS' in source,
}

for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {label}")

assert all(checks.values()), "A verification check failed."
print("\nAll checks passed. No training was started.")


In [ ]:

from pathlib import Path
import json
import os
import subprocess
import sys

project = Path("/kaggle/working/small-model-alignment-starter")
env = os.environ.copy()

env.update({
    "SFT_MAX_TRAIN_EXAMPLES": "8",
    "SFT_MAX_LENGTH": "128",
    "SFT_BATCH_SIZE": "2",
    "SFT_GRAD_ACCUM": "2",
    "SFT_EPOCHS": "1",
    "SFT_VAL_BATCHES": "2",
    "SFT_SAVE_EVERY": "0",
    "SFT_RESUME": "0",
    "SFT_DISABLE_CHECKPOINTS": "1",
})

result = subprocess.run(
    [sys.executable, "-u", "train_sft.py"],
    cwd=project,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=900,
)

print(result.stdout[-12000:])
print("\nSmoke-test exit code:", result.returncode)

# Check whether the evaluation metrics survived.
metrics_path = project / "outputs" / "sft_run" / "metrics.jsonl"
if metrics_path.exists():
    print("\nMetrics file contents:")
    print(metrics_path.read_text(encoding="utf-8"))
else:
    print("\nMetrics file not found.")

assert result.returncode == 0, "Smoke test failed; do not start full training."
assert metrics_path.exists(), "Evaluation metrics were not written."

records = [
    json.loads(line)
    for line in metrics_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
assert records, "Metrics file is empty."
assert "validation_response_token_loss" in records[-1]

print("\nPASS: training, validation, and metrics persistence work.")
print("No checkpoint files were written during the smoke test.")


In [ ]:

from pathlib import Path
import ast

project = Path("/kaggle/working/small-model-alignment-starter")
script_path = project / "train_sft.py"
source = script_path.read_text(encoding="utf-8")

# Remove the deprecation warning.
source = source.replace(
    "torch_dtype=torch.float32",
    "dtype=torch.float32",
)

# Store a JSON null when there is no previous best validation loss.
source = source.replace(
    '"best_validation_loss_before_update": best_val_loss,',
    '"best_validation_loss_before_update": '
    '(best_val_loss if best_val_loss != float("inf") else None),',
)

# Verify syntax before writing the updated script.
ast.parse(source)
script_path.write_text(source, encoding="utf-8")

# Remove only the smoke-test metrics, so full-run metrics start clean.
metrics_path = project / "outputs" / "sft_run" / "metrics.jsonl"
if metrics_path.exists():
    metrics_path.unlink()

# Verify the changes.
updated = script_path.read_text(encoding="utf-8")
assert "torch_dtype=torch.float32" not in updated
assert '"best_validation_loss_before_update": (best_val_loss if best_val_loss != float("inf") else None),' in updated

print("PASS — Script syntax is valid.")
print("PASS — Deprecated dtype argument removed.")
print("PASS — First evaluation will use JSON null instead of Infinity.")
print("PASS — Smoke-test metrics cleared.")
print("\nNo training was started.")


In [ ]:

from pathlib import Path
import shutil
import ast
import re

project = Path("/kaggle/working/small-model-alignment-starter")
script_path = project / "train_sft.py"

# 1. Check available disk space.
usage = shutil.disk_usage("/kaggle/working")
print("DISK SPACE")
print(f"Total:     {usage.total / 1024**3:.2f} GiB")
print(f"Used:      {usage.used / 1024**3:.2f} GiB")
print(f"Available: {usage.free / 1024**3:.2f} GiB")

# 2. Check that the required data files exist.
train_path = project / "data" / "sft" / "train.json"
val_path = project / "data" / "sft" / "validation.json"

print("\nDATA FILES")
for path in [train_path, val_path]:
    print(f"{'PASS' if path.exists() else 'MISSING'} — {path}")
    assert path.exists(), f"Required dataset missing: {path}"

# 3. Verify important settings in the training script.
source = script_path.read_text(encoding="utf-8")
ast.parse(source)

print("\nSCRIPT SETTINGS")
for pattern, label in [
    (r'MODEL_NAME\s*=\s*.*Qwen2\.5-0\.5B', "Qwen2.5-0.5B base model"),
    (r'MAX_LENGTH\s*=\s*512', "Context length = 512"),
    (r'BATCH_SIZE\s*=\s*2', "Batch size = 2"),
    (r'GRAD_ACCUM\s*=\s*16', "Gradient accumulation = 16"),
    (r'EPOCHS\s*=\s*1', "One training epoch"),
    (r'LR\s*=\s*2e-5', "Learning rate = 2e-5"),
]:
    passed = bool(re.search(pattern, source))
    print(f"{'PASS' if passed else 'CHECK'} — {label}")

# 4. Inspect output-directory contents without deleting anything.
output_dir = project / "outputs" / "sft_run"
print("\nOUTPUT DIRECTORY")
if output_dir.exists():
    for item in sorted(output_dir.iterdir()):
        print(item.name)
else:
    print("Will be created when training starts.")

print("\nDisk check complete. No training was started.")


In [ ]:

from pathlib import Path

script_path = Path(
    "/kaggle/working/small-model-alignment-starter/train_sft.py"
)
lines = script_path.read_text(encoding="utf-8").splitlines()

keywords = [
    "MODEL_NAME",
    "MAX_LENGTH",
    "BATCH_SIZE",
    "GRAD_ACCUM",
    "EPOCHS",
    "LR",
    "WEIGHT_DECAY",
    "WARMUP_RATIO",
    "SAVE_EVERY",
    "VAL_BATCHES",
    "OUTPUT_DIR",
    "RESUME",
    "DISABLE_CHECKPOINTS",
]

print("Relevant configuration and checkpoint lines:\n")

for i, line in enumerate(lines, start=1):
    if any(keyword in line for keyword in keywords):
        print(f"{i:03}: {line}")

print("\nInspection complete. No training was started.")


In [ ]:

from pathlib import Path
import ast

project = Path("/kaggle/working/small-model-alignment-starter")
script_path = project / "train_sft.py"
source = script_path.read_text(encoding="utf-8")

old_device = 'DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")'
new_device = 'DEVICE = torch.device(os.environ.get("SFT_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"))'

assert old_device in source, "Expected device-selection line was not found."
source = source.replace(old_device, new_device, 1)

old_model_load = '''    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float32,
    ).to(DEVICE)'''

new_model_load = '''    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float32,
    ).to(DEVICE)

    # Reduce activation memory during training.
    model.gradient_checkpointing_enable()
    model.config.use_cache = False'''

assert old_model_load in source, "Expected model-loading block was not found."
source = source.replace(old_model_load, new_model_load, 1)

ast.parse(source)
script_path.write_text(source, encoding="utf-8")

updated = script_path.read_text(encoding="utf-8")
assert 'os.environ.get("SFT_DEVICE"' in updated
assert "model.gradient_checkpointing_enable()" in updated
assert "model.config.use_cache = False" in updated

print("PASS — Device can be selected using SFT_DEVICE.")
print("PASS — Gradient checkpointing enabled.")
print("PASS — KV cache disabled for training.")
print("PASS — Python syntax is valid.")
print("\nNo training was started.")


In [ ]:

from pathlib import Path
import os
import subprocess
import sys
import json

project = Path("/kaggle/working/small-model-alignment-starter")
output_dir = project / "outputs" / "sft_run"
metrics_path = output_dir / "metrics.jsonl"

# Clear only the smoke-test metrics, not datasets or model files.
if metrics_path.exists():
    metrics_path.unlink()

env = os.environ.copy()
env.update({
    "SFT_DEVICE": "cuda:1",
    "SFT_MAX_TRAIN_EXAMPLES": "8",
    "SFT_MAX_LENGTH": "512",
    "SFT_BATCH_SIZE": "1",
    "SFT_GRAD_ACCUM": "2",
    "SFT_EPOCHS": "1",
    "SFT_VAL_BATCHES": "2",
    "SFT_SAVE_EVERY": "0",
    "SFT_RESUME": "0",
    "SFT_DISABLE_CHECKPOINTS": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})

result = subprocess.run(
    [sys.executable, "-u", "train_sft.py"],
    cwd=project,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=900,
)

print(result.stdout[-12000:])
print("\nSmoke-test exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "GPU 1 smoke test failed. Do not launch full training yet."
    )

assert metrics_path.exists(), "Metrics file was not written."
records = [
    json.loads(line)
    for line in metrics_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
assert records, "Metrics file is empty."
assert "validation_response_token_loss" in records[-1]

print("\nPASS — GPU 1 smoke test completed.")
print("Validation metrics were saved.")
print("No checkpoint files were written.")


In [ ]:

from pathlib import Path
import os
import subprocess
import sys
import json
import time

project = Path("/kaggle/working/small-model-alignment-starter")
output_dir = project / "outputs" / "sft_run"
output_dir.mkdir(parents=True, exist_ok=True)

log_path = output_dir / "train.log"
pid_path = output_dir / "train.pid"
status_path = output_dir / "run_status.json"
metrics_path = output_dir / "metrics.jsonl"

# Check the previous PID carefully; a defunct process is not active training.
if pid_path.exists():
    old_pid = int(pid_path.read_text().strip())
    check = subprocess.run(
        ["ps", "-p", str(old_pid), "-o", "stat=,args="],
        capture_output=True,
        text=True,
    )
    old_info = check.stdout.strip()

    if old_info and not old_info.split()[0].startswith("Z"):
        if "train_sft.py" in old_info:
            raise RuntimeError(
                f"Training may already be running under PID {old_pid}. "
                "Do not launch a second run."
            )
        print("Previous PID belongs to another process; it will not be stopped.")
    else:
        print("Previous training PID is stale or defunct; safe to replace.")

# Remove only the metrics generated by the smoke test.
if metrics_path.exists():
    metrics_path.unlink()

env = os.environ.copy()

# Remove smoke-test-only limits if they exist in the notebook environment.
env.pop("SFT_MAX_TRAIN_EXAMPLES", None)

env.update({
    "SFT_DEVICE": "cuda:1",
    "SFT_MAX_LENGTH": "512",
    "SFT_BATCH_SIZE": "1",
    "SFT_GRAD_ACCUM": "32",
    "SFT_EPOCHS": "1",
    "SFT_LR": "2e-5",
    "SFT_VAL_BATCHES": "0",
    "SFT_SAVE_EVERY": "100",
    "SFT_RESUME": "0",
    "SFT_DISABLE_CHECKPOINTS": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})

config = {
    "model": "Qwen/Qwen2.5-0.5B",
    "training_examples": 30000,
    "validation_examples": 1000,
    "device": "cuda:1",
    "context_length": 512,
    "batch_size": 1,
    "gradient_accumulation": 32,
    "effective_batch_size": 32,
    "learning_rate": 2e-5,
    "epochs": 1,
    "gradient_checkpointing": True,
    "checkpoint_interval_optimizer_steps": 100,
}
status_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        [sys.executable, "-u", "train_sft.py"],
        cwd=project,
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
        start_new_session=True,
    )

pid_path.write_text(str(process.pid), encoding="utf-8")
time.sleep(8)

return_code = process.poll()
print("Training PID:", process.pid)
print("Log:", log_path)

if return_code is None:
    print("Status: RUNNING")
    print("\nStartup log:")
    print("\n".join(log_path.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]))
else:
    print("Status: EXITED")
    print("Exit code:", return_code)
    print("\nStartup log:")
    print("\n".join(log_path.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-50:]))
    raise RuntimeError("Training did not remain running. Inspect the log.")

print("\nFull-run configuration:")
print(json.dumps(config, indent=2))


In [ ]:

from pathlib import Path
import subprocess
import shutil
import json

project = Path("/kaggle/working/small-model-alignment-starter")
output_dir = project / "outputs" / "sft_run"

# Read recent training logs.
log_path = output_dir / "train.log"
print("=== RECENT TRAINING LOG ===")
if log_path.exists():
    lines = log_path.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
    print("\n".join(lines[-30:]) if lines else "Log is still empty.")
else:
    print("Log file not found.")

# Check whether PID 695 is running.
print("\n=== PROCESS STATUS ===")
result = subprocess.run(
    ["ps", "-p", "695", "-o", "pid=,stat=,etime=,cmd="],
    capture_output=True,
    text=True,
)
print(result.stdout.strip() or "PID 695 is no longer running.")

# Check GPU memory and utilization.
print("\n=== GPU STATUS ===")
gpu = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
)
print(gpu.stdout if gpu.returncode == 0 else gpu.stderr)

# Check disk space and validation metrics.
usage = shutil.disk_usage("/kaggle/working")
print(f"\nFree disk space: {usage.free / 1024**3:.2f} GiB")

metrics_path = output_dir / "metrics.jsonl"
print("\n=== VALIDATION METRICS ===")
if metrics_path.exists():
    for line in metrics_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            print(json.loads(line))
else:
    print("No validation metrics yet; the first evaluation occurs after the epoch.")


In [ ]:
from pathlib import Path
import os

PROJECT = Path("/kaggle/working/small-model-alignment-starter")
SFT_DIR = PROJECT / "outputs" / "sft_run"

print("SFT output directory:", SFT_DIR)
print("Directory exists:", SFT_DIR.exists())

print("\nFiles and folders:")
for path in sorted(SFT_DIR.rglob("*")):
    if path.is_file():
        size_gb = path.stat().st_size / (1024 ** 3)
        print(f"FILE  {size_gb:6.2f} GB  {path.relative_to(SFT_DIR)}")
    elif path.is_dir():
        print(f"DIR              {path.relative_to(SFT_DIR)}")

print("\nRequired model directories:")
for name in ["best_checkpoint", "final_checkpoint"]:
    path = SFT_DIR / name
    print(f"\n{name}: exists={path.is_dir()}")
    if path.is_dir():
        for f in sorted(path.iterdir()):
            if f.is_file():
                print(f"  {f.name}: {f.stat().st_size / (1024**3):.2f} GB")

print("\nMetrics file exists:",
      (SFT_DIR / "metrics.jsonl").is_file())

print("\nFree disk space:",
      f"{os.statvfs('/kaggle/working').f_bavail * os.statvfs('/kaggle/working').f_frsize / (1024**3):.2f} GB")

In [ ]:
from pathlib import Path
import zipfile
import os

PROJECT = Path("/kaggle/working/small-model-alignment-starter")
SFT_DIR = PROJECT / "outputs" / "sft_run"
BACKUP_DIR = Path("/kaggle/working/recovery_backups")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

BACKUP_ZIP = BACKUP_DIR / "sft_completed_backup.zip"

# Include both model checkpoints and the training records.
included = [
    SFT_DIR / "best_checkpoint",
    SFT_DIR / "final_checkpoint",
]

for filename in ["metrics.jsonl", "train.log", "run_status.json"]:
    path = SFT_DIR / filename
    if path.is_file():
        included.append(path)

# Write without recompressing model weights; this is faster and avoids
# wasting time trying to compress already-compressed safetensors files.
with zipfile.ZipFile(
    BACKUP_ZIP, "w", compression=zipfile.ZIP_STORED
) as archive:
    for item in included:
        if item.is_dir():
            for file in item.rglob("*"):
                if file.is_file():
                    archive.write(file, file.relative_to(PROJECT))
        elif item.is_file():
            archive.write(item, item.relative_to(PROJECT))

# Verify archive integrity.
with zipfile.ZipFile(BACKUP_ZIP, "r") as archive:
    bad_file = archive.testzip()
    print("Archive integrity:", "PASS" if bad_file is None else f"FAILED: {bad_file}")
    print("Files in archive:", len(archive.namelist()))

print("Backup path:", BACKUP_ZIP)
print(f"Backup size: {BACKUP_ZIP.stat().st_size / (1024**3):.2f} GB")
print(f"Free disk space: {os.statvfs('/kaggle/working').f_bavail * os.statvfs('/kaggle/working').f_frsize / (1024**3):.2f} GB")

In [ ]:
import os
from pathlib import Path
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

backup = Path("recovery_backups/sft_completed_backup.zip")

print("Backup exists:", backup.exists())

if backup.exists():
    print(f"Size: {backup.stat().st_size / (1024**3):.2f} GB")
    display(FileLink("recovery_backups/sft_completed_backup.zip"))
else:
    print("Backup not found. Do not stop the session.")

In [ ]:

import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".zip"):
            print(os.path.join(root, file))


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(f"\nFolder: {root}")
    for file in files:
        print(f"  {file}")

In [ ]:

import os
import json
from pathlib import Path

RUN_DIR = Path(
    "/kaggle/input/datasets/khadijawn/"
    "stf-backup/outputs/sft_run"
)

# 1. Check the saved run status
status_file = RUN_DIR / "run_status.json"
print("=== SFT RUN STATUS ===")

with open(status_file, "r") as f:
    print(json.dumps(json.load(f), indent=2))

# 2. Show the latest training metrics
metrics_file = RUN_DIR / "metrics.jsonl"
print("\n=== LAST 5 METRIC RECORDS ===")

with open(metrics_file, "r") as f:
    lines = f.readlines()

for line in lines[-5:]:
    print(line.strip())

# 3. Show the last lines of the training log
log_file = RUN_DIR / "train.log"
print("\n=== LAST 20 TRAINING LOG LINES ===")

with open(log_file, "r", errors="replace") as f:
    log_lines = f.readlines()

for line in log_lines[-20:]:
    print(line.rstrip())

# 4. Check available working storage
import shutil

total, used, free = shutil.disk_usage("/kaggle/working")
print("\n=== KAGGLE WORKING STORAGE ===")
print(f"Total: {total / 1024**3:.2f} GiB")
print(f"Used:  {used / 1024**3:.2f} GiB")
print(f"Free:  {free / 1024**3:.2f} GiB")


In [ ]:

import os

print("=== FILES AVAILABLE IN KAGGLE INPUT ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        path = os.path.join(root, file)
        print(path)


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:

import os
import hashlib

BASES = [
    "/kaggle/input/datasets/khadijawn/alignment-asssignment-recovery",
    "/kaggle/input/datasets/khadijawn/aa-step8",
]

for base in BASES:
    adapter = os.path.join(base, "submission_adapter.py")
    print("\n" + "=" * 65)
    print("REPOSITORY:", base)
    
    if os.path.isfile(adapter):
        with open(adapter, "rb") as f:
            content = f.read()
        print("Adapter size:", len(content), "bytes")
        print("Adapter SHA256:", hashlib.sha256(content).hexdigest())
        print("\nAdapter source:\n")
        print(content.decode("utf-8")[:12000])
    else:
        print("submission_adapter.py not found")

    print("\nRelevant data files:")
    for root, dirs, files in os.walk(base):
        for name in files:
            lower = name.lower()
            if any(word in lower for word in [
                "preference", "ultrafeedback", "hh_rlhf",
                "chosen", "rejected", "adapter"
            ]):
                path = os.path.join(root, name)
                print(path, "|", os.path.getsize(path), "bytes")


In [ ]:

import os
import sys
import shutil
import subprocess

SOURCE = "/kaggle/input/datasets/khadijawn/alignment-asssignment-recovery"
WORKING = "/kaggle/working/alignment-assignment-check"

# Make a writable copy of the repository.
if os.path.exists(WORKING):
    shutil.rmtree(WORKING)

shutil.copytree(SOURCE, WORKING)

print("=== DATA FILES ===")
for folder in ["data", "eval_sets", "alignment_pre_sft_backup/data"]:
    path = os.path.join(WORKING, folder)
    if os.path.isdir(path):
        print(f"\n{folder}/")
        for root, dirs, files in os.walk(path):
            for name in files:
                full_path = os.path.join(root, name)
                print(
                    os.path.relpath(full_path, WORKING),
                    f"({os.path.getsize(full_path):,} bytes)"
                )

print("\n=== RUNNING PUBLIC TESTS ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"],
    cwd=WORKING,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)
print("Pytest exit code:", result.returncode)


In [ ]:

from datasets import load_dataset

datasets_to_check = [
    ("HH-RLHF", "Anthropic/hh-rlhf", "train"),
    (
        "UltraFeedback",
        "HuggingFaceH4/ultrafeedback_binarized",
        "train_prefs",
    ),
]

for name, repo_id, split in datasets_to_check:
    print(f"\n{'=' * 60}")
    print(name)
    print("Dataset:", repo_id)
    print("Split:", split)

    try:
        stream = load_dataset(
            repo_id,
            split=split,
            streaming=True,
        )
        example = next(iter(stream))

        print("Status: accessible")
        print("Columns:", list(example.keys()))

        for field in ("prompt", "chosen", "rejected"):
            if field in example:
                value = example[field]
                preview = repr(value)
                print(f"\n{field}: {preview[:350]}")

    except Exception as error:
        print("Status: failed")
        print(type(error).__name__, str(error)[:400])


In [ ]:

import json
from pathlib import Path

from datasets import load_dataset

OUTPUT_DIR = Path("/kaggle/working/alignment-assignment-check/data/dpo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SIZE = 15_000
VAL_SIZE = 1_000
TOTAL_SIZE = TRAIN_SIZE + VAL_SIZE
SEED = 42

HH_MARKER = "\n\nAssistant: "


def prepare_hh(row):
    """Extract the final assistant response from an HH-RLHF pair."""
    chosen = row["chosen"]
    rejected = row["rejected"]

    chosen_pos = chosen.rfind(HH_MARKER)
    rejected_pos = rejected.rfind(HH_MARKER)

    if chosen_pos < 0 or rejected_pos < 0:
        return None

    chosen_prefix = chosen[:chosen_pos + len(HH_MARKER)]
    rejected_prefix = rejected[:rejected_pos + len(HH_MARKER)]

    # Both responses must follow the same conversation prompt.
    if chosen_prefix != rejected_prefix:
        return None

    chosen_response = chosen[chosen_pos + len(HH_MARKER):].strip()
    rejected_response = rejected[rejected_pos + len(HH_MARKER):].strip()

    if not chosen_response or not rejected_response:
        return None

    return {
        "prompt": chosen_prefix.rstrip(),
        "chosen": chosen_response,
        "rejected": rejected_response,
    }


def last_assistant_response(messages):
    """Return the content of the last assistant message."""
    for message in reversed(messages):
        if message.get("role") == "assistant":
            content = message.get("content", "")
            if isinstance(content, str) and content.strip():
                return content.strip()
    return None


def prepare_ultrafeedback(row):
    """Convert UltraFeedback message lists into text preference pairs."""
    prompt = row.get("prompt", "")
    chosen = last_assistant_response(row.get("chosen", []))
    rejected = last_assistant_response(row.get("rejected", []))

    if not isinstance(prompt, str) or not prompt.strip():
        return None
    if not chosen or not rejected:
        return None

    return {
        "prompt": prompt.strip(),
        "chosen": chosen,
        "rejected": rejected,
    }


def save_dataset(dataset_name, repo_id, split, prepare_row, seed):
    print(f"\nPreparing {dataset_name}...")

    stream = load_dataset(
        repo_id,
        split=split,
        streaming=True,
    ).shuffle(seed=seed, buffer_size=20_000)

    rows = []
    skipped = 0

    for raw_row in stream:
        prepared = prepare_row(raw_row)

        if prepared is None:
            skipped += 1
            continue

        rows.append(prepared)

        if len(rows) == TOTAL_SIZE:
            break

    if len(rows) != TOTAL_SIZE:
        raise RuntimeError(
            f"{dataset_name}: expected {TOTAL_SIZE} valid rows, "
            f"but collected {len(rows)}. No complete split was saved."
        )

    train_rows = rows[:TRAIN_SIZE]
    val_rows = rows[TRAIN_SIZE:]

    train_path = OUTPUT_DIR / f"{dataset_name}_train.json"
    val_path = OUTPUT_DIR / f"{dataset_name}_validation.json"

    with train_path.open("w", encoding="utf-8") as f:
        json.dump(train_rows, f, ensure_ascii=False)

    with val_path.open("w", encoding="utf-8") as f:
        json.dump(val_rows, f, ensure_ascii=False)

    print("Training examples:", len(train_rows))
    print("Validation examples:", len(val_rows))
    print("Skipped invalid rows:", skipped)
    print("Saved:", train_path)
    print("Saved:", val_path)
    print("Example prompt:", repr(train_rows[0]["prompt"][:180]))
    print("Example chosen:", repr(train_rows[0]["chosen"][:180]))
    print("Example rejected:", repr(train_rows[0]["rejected"][:180]))


save_dataset(
    dataset_name="hh_rlhf",
    repo_id="Anthropic/hh-rlhf",
    split="train",
    prepare_row=prepare_hh,
    seed=SEED,
)

save_dataset(
    dataset_name="ultrafeedback",
    repo_id="HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs",
    prepare_row=prepare_ultrafeedback,
    seed=SEED,
)

print("\nDataset preparation finished.")
print("No model training was started.")


In [ ]:

import json
from pathlib import Path

DATA_DIR = Path("/kaggle/working/alignment-assignment-check/data/dpo")

for name in ["hh_rlhf", "ultrafeedback"]:
    print(f"\n{'=' * 60}")
    print(name.upper())

    for split, expected_count in [
        ("train", 15_000),
        ("validation", 1_000),
    ]:
        path = DATA_DIR / f"{name}_{split}.json"

        with path.open("r", encoding="utf-8") as f:
            rows = json.load(f)

        assert len(rows) == expected_count, (
            f"{path.name}: expected {expected_count}, got {len(rows)}"
        )

        required = {"prompt", "chosen", "rejected"}
        assert all(required.issubset(row) for row in rows), (
            f"{path.name}: missing required fields"
        )

        empty = sum(
            not row["prompt"].strip()
            or not row["chosen"].strip()
            or not row["rejected"].strip()
            for row in rows
        )

        duplicate_pairs = len(rows) - len({
            (row["prompt"], row["chosen"], row["rejected"])
            for row in rows
        })

        print(f"\n{split}: {len(rows)} rows")
        print("Empty/incomplete examples:", empty)
        print("Duplicate full pairs:", duplicate_pairs)

        if split == "validation":
            for i, row in enumerate(rows[:3], start=1):
                print(f"\nExample {i}")
                print("Prompt:", repr(row["prompt"][:200]))
                print("Chosen:", repr(row["chosen"][:200]))
                print("Rejected:", repr(row["rejected"][:200]))

print("\nDataset validation finished.")


In [ ]:

from pathlib import Path
import torch

# 1. Check GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(
        f"  Total memory: "
        f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB"
    )

# 2. Check the saved SFT checkpoints
checkpoint_paths = {
    "Best SFT checkpoint": Path(
        "/kaggle/input/datasets/khadijawn/stf-backup/"
        "outputs/sft_run/best_checkpoint"
    ),
    "Final SFT checkpoint": Path(
        "/kaggle/input/datasets/khadijawn/stf-backup/"
        "outputs/sft_run/final_checkpoint"
    ),
}

for label, path in checkpoint_paths.items():
    print(f"\n{label}")
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        files = sorted(p.name for p in path.iterdir())
        print("Files:", files[:30])
        print("Total files:", len(files))

# 3. Check repository files
repo = Path("/kaggle/working/alignment-assignment-check")
print("\nRepository exists:", repo.exists())

if repo.exists():
    for relative in [
        "submission_adapter.py",
        "README.md",
        "tests",
        "data/dpo/hh_rlhf_train.json",
        "data/dpo/ultrafeedback_train.json",
    ]:
        path = repo / relative
        print(f"{relative}: {'exists' if path.exists() else 'missing'}")

print("\nEnvironment check complete. No training started.")


In [ ]:

import inspect
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = Path("/kaggle/working/alignment-assignment-check")
SFT_PATH = (
    "/kaggle/input/datasets/khadijawn/stf-backup/"
    "outputs/sft_run/best_checkpoint"
)
BASE_MODEL = "Qwen/Qwen2.5-0.5B"

# 1. Inspect the existing adapter helper signatures.
import sys
sys.path.insert(0, str(REPO))

import submission_adapter as adapter

print("Existing adapter functions:")
for name in [
    "_build_lm_batch",
    "build_dpo_batch",
    "compute_response_logprobs",
    "compute_dpo_loss_from_logps",
    "compute_dpo_loss",
]:
    fn = getattr(adapter, name, None)
    if fn is not None:
        print(f"{name}{inspect.signature(fn)}")

# 2. Load the tokenizer.
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocabulary size:", len(tokenizer))
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token ID:", tokenizer.pad_token_id)

# 3. Load the trainable policy on GPU 0 in FP32.
print("\nLoading policy on GPU 0 in FP32...")
policy = AutoModelForCausalLM.from_pretrained(
    SFT_PATH,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)
policy = policy.to("cuda:0")
policy.eval()

# 4. Load the frozen reference on GPU 1 in FP16.
print("Loading reference on GPU 1 in FP16...")
reference = AutoModelForCausalLM.from_pretrained(
    SFT_PATH,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
reference = reference.to("cuda:1")
reference.eval()

for parameter in reference.parameters():
    parameter.requires_grad_(False)

print("\nPolicy device:", next(policy.parameters()).device)
print("Policy dtype:", next(policy.parameters()).dtype)
print("Reference device:", next(reference.parameters()).device)
print("Reference dtype:", next(reference.parameters()).dtype)
print(
    "Reference trainable parameters:",
    sum(p.numel() for p in reference.parameters() if p.requires_grad),
)

# 5. Run a short forward pass through both models.
test_text = "Human: What is the capital of France?\nAssistant:"
encoded = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=128,
)
policy_inputs = {
    key: value.to("cuda:0")
    for key, value in encoded.items()
}
reference_inputs = {
    key: value.to("cuda:1")
    for key, value in encoded.items()
}

with torch.no_grad():
    policy_logits = policy(**policy_inputs).logits
    reference_logits = reference(**reference_inputs).logits

print("\nPolicy logits shape:", tuple(policy_logits.shape))
print("Reference logits shape:", tuple(reference_logits.shape))
print("Policy logits finite:", torch.isfinite(policy_logits).all().item())
print(
    "Reference logits finite:",
    torch.isfinite(reference_logits).all().item(),
)

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved = torch.cuda.memory_reserved(i) / 1024**3
    print(
        f"GPU {i}: allocated={allocated:.2f} GiB, "
        f"reserved={reserved:.2f} GiB"
    )

assert torch.isfinite(policy_logits).all(), "Policy produced non-finite logits."
assert torch.isfinite(reference_logits).all(), (
    "Reference produced non-finite logits."
)

print("\nSmoke test passed. No training or optimizer steps were run.")


In [ ]:

import inspect
import submission_adapter as adapter

for name in [
    "_build_lm_batch",
    "build_dpo_batch",
    "compute_response_logprobs",
    "compute_dpo_loss_from_logps",
    "compute_dpo_loss",
]:
    fn = getattr(adapter, name)
    print(f"\n{'=' * 25} {name} {'=' * 25}")
    print(inspect.getsource(fn))


In [ ]:

import json
import torch
from pathlib import Path

BETA = 0.1
MAX_LENGTH = 64

data_path = Path(
    "/kaggle/working/alignment-assignment-check/"
    "data/dpo/hh_rlhf_train.json"
)

with data_path.open("r", encoding="utf-8") as f:
    train_rows = json.load(f)

# Use one real preference pair for the smoke test.
row = train_rows[0]
prompts = [row["prompt"]]
chosen = [row["chosen"]]
rejected = [row["rejected"]]

print("Prompt:", repr(prompts[0][:150]))
print("Chosen:", repr(chosen[0][:150]))
print("Rejected:", repr(rejected[0][:150]))

# Keep the policy in training mode so gradients can be computed.
policy.train()
policy.zero_grad(set_to_none=True)

# Policy log probabilities: GPU 0, with gradients enabled.
policy_chosen_logps = adapter.compute_response_logprobs(
    policy, tokenizer, prompts, chosen, MAX_LENGTH
)
policy_rejected_logps = adapter.compute_response_logprobs(
    policy, tokenizer, prompts, rejected, MAX_LENGTH
)

# Reference log probabilities: GPU 1, without gradients.
reference.eval()
with torch.no_grad():
    reference_chosen_logps = adapter.compute_response_logprobs(
        reference, tokenizer, prompts, chosen, MAX_LENGTH
    )
    reference_rejected_logps = adapter.compute_response_logprobs(
        reference, tokenizer, prompts, rejected, MAX_LENGTH
    )

# Move only the small score tensors to the policy's GPU.
reference_chosen_logps = reference_chosen_logps.to("cuda:0")
reference_rejected_logps = reference_rejected_logps.to("cuda:0")

# Compute the DPO loss.
loss = adapter.compute_dpo_loss_from_logps(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps,
    BETA,
)

# Calculate the standard DPO rewards and metrics.
with torch.no_grad():
    chosen_rewards = BETA * (
        policy_chosen_logps.detach() - reference_chosen_logps
    )
    rejected_rewards = BETA * (
        policy_rejected_logps.detach() - reference_rejected_logps
    )

    reward_margin = (chosen_rewards - rejected_rewards).mean()
    reward_accuracy = (
        chosen_rewards > rejected_rewards
    ).float().mean()

print("\n--- DPO smoke-test results ---")
print("Policy chosen log-probability:", policy_chosen_logps.item())
print("Policy rejected log-probability:", policy_rejected_logps.item())
print("Reference chosen log-probability:", reference_chosen_logps.item())
print("Reference rejected log-probability:", reference_rejected_logps.item())
print("DPO loss:", loss.item())
print("Chosen reward:", chosen_rewards.item())
print("Rejected reward:", rejected_rewards.item())
print("Reward margin:", reward_margin.item())
print("Reward accuracy:", reward_accuracy.item())

assert torch.isfinite(loss).item(), "DPO loss is not finite."
assert torch.isfinite(reward_margin).item(), "Reward margin is not finite."

# Test gradient flow without updating any parameters.
loss.backward()

grad_params = [
    p for p in policy.parameters()
    if p.requires_grad and p.grad is not None
]
assert grad_params, "No gradients reached the policy."

gradients_finite = all(
    torch.isfinite(p.grad).all().item()
    for p in grad_params
)
assert gradients_finite, "Policy gradients contain non-finite values."

print("Parameters receiving gradients:", len(grad_params))
print("All policy gradients finite:", gradients_finite)
print("Reference has gradients:", any(
    p.grad is not None for p in reference.parameters()
))

# Discard test gradients. No optimizer step was taken.
policy.zero_grad(set_to_none=True)
policy.eval()

print("\nSmoke test passed. No model weights were updated.")


In [ ]:

# ============================================================
# COMPLETE DPO TRAINING CELL — MEMORY-EFFICIENT VERSION
# ============================================================

import gc
import json
import random
from pathlib import Path

import torch
import torch.nn.functional as F


# -----------------------------
# 1. Configuration
# -----------------------------

BETA = 0.1
MAX_LENGTH = 512
MAX_PROMPT_TOKENS = 256
GRAD_ACCUM_STEPS = 32
LEARNING_RATE = 1e-6
MAX_GRAD_NORM = 1.0
SEED = 42

REPO = Path("/kaggle/working/alignment-assignment-check")

POLICY_DEVICE = torch.device("cuda:0")
REFERENCE_DEVICE = torch.device("cuda:1")

assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() >= 2, "Two GPUs are required."

gc.collect()
torch.cuda.empty_cache()

# The reference must never be trained.
reference.eval()
for parameter in reference.parameters():
    parameter.requires_grad_(False)

# Save memory by recomputing intermediate activations during backward.
policy.config.use_cache = False
policy.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

print("Policy device:", next(policy.parameters()).device)
print("Reference device:", next(reference.parameters()).device)
print("Gradient checkpointing enabled.")
print("DPO beta:", BETA)
print("Learning rate:", LEARNING_RATE)


# -----------------------------
# 2. Prepare one preference pair
# -----------------------------

def make_dpo_pair_batch(tokenizer, row, max_length=MAX_LENGTH):
    """
    Construct a batch containing two sequences:
      sequence 0 = prompt + chosen response
      sequence 1 = prompt + rejected response

    Prompt tokens are excluded from the response-only loss.
    Padding tokens are also excluded.
    """

    pad_id = tokenizer.pad_token_id
    eos_id = tokenizer.eos_token_id

    if pad_id is None:
        raise ValueError("Tokenizer must have a pad token.")
    if eos_id is None:
        raise ValueError("Tokenizer must have an EOS token.")

    prompt_ids = tokenizer.encode(
        row["prompt"], add_special_tokens=False
    )

    # Retain the end of the prompt if it is too long.
    prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]
    prompt_ids = prompt_ids[:max_length - 1]

    sequences = []

    for response in (row["chosen"], row["rejected"]):
        response_ids = tokenizer.encode(
            response, add_special_tokens=False
        )

        # Reserve one position for EOS.
        response_budget = max_length - len(prompt_ids) - 1
        response_ids = response_ids[:response_budget]
        response_ids = response_ids + [eos_id]

        input_ids = prompt_ids + response_ids

        # Ignore the prompt; learn only from the response and EOS.
        labels = (
            [-100] * len(prompt_ids)
            + response_ids.copy()
        )

        sequences.append((input_ids, labels))

    width = max(len(ids) for ids, _ in sequences)

    padded_ids = []
    padded_masks = []
    padded_labels = []

    for input_ids, labels in sequences:
        padding_length = width - len(input_ids)

        padded_ids.append(
            input_ids + [pad_id] * padding_length
        )
        padded_masks.append(
            [1] * len(input_ids) + [0] * padding_length
        )
        padded_labels.append(
            labels + [-100] * padding_length
        )

    return {
        "input_ids": torch.tensor(
            padded_ids, dtype=torch.long
        ),
        "attention_mask": torch.tensor(
            padded_masks, dtype=torch.long
        ),
        "labels": torch.tensor(
            padded_labels, dtype=torch.long
        ),
    }


# -----------------------------
# 3. Memory-efficient response log probabilities
# -----------------------------

def get_pair_logps(model, batch, chunk_size=4):
    """
    Return two summed response log probabilities:
      [chosen_log_probability, rejected_log_probability]

    The LM head is applied only at response-token positions,
    and vocabulary logits are calculated in small chunks.
    """

    outputs = model.model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        use_cache=False,
    )

    # Causal LM: hidden state at t predicts token at t+1.
    hidden = outputs.last_hidden_state[:, :-1, :]
    labels = batch["labels"][:, 1:]

    valid = labels.ne(-100)
    token_counts = valid.sum(dim=1).tolist()

    selected_hidden = hidden[valid]
    selected_labels = labels[valid]

    if selected_labels.numel() == 0:
        raise ValueError("This pair has no response tokens.")

    # Release references that are no longer needed.
    del outputs, hidden, labels, valid

    token_logps = []

    for start in range(0, selected_labels.numel(), chunk_size):
        end = min(start + chunk_size, selected_labels.numel())

        logits = model.lm_head(
            selected_hidden[start:end]
        ).float()

        target_logits = logits.gather(
            dim=-1,
            index=selected_labels[start:end].unsqueeze(-1),
        ).squeeze(-1)

        log_normalizer = torch.logsumexp(logits, dim=-1)
        chunk_logps = target_logits - log_normalizer

        token_logps.append(chunk_logps)

        del logits, target_logits, log_normalizer, chunk_logps

    token_logps = torch.cat(token_logps)

    # Sum response-token log probabilities separately for each sequence.
    pair_logps = []
    offset = 0

    for count in token_counts:
        pair_logps.append(
            token_logps[offset:offset + count].sum()
        )
        offset += count

    result = torch.stack(pair_logps)

    del selected_hidden, selected_labels, token_logps
    return result


# -----------------------------
# 4. DPO loss and reward metrics
# -----------------------------

def dpo_pair_metrics(
    policy_logps,
    reference_logps,
    beta=BETA,
):
    """
    policy_logps:    [chosen_logp, rejected_logp]
    reference_logps: [chosen_logp, rejected_logp]
    """

    policy_chosen = policy_logps[0]
    policy_rejected = policy_logps[1]

    reference_chosen = reference_logps[0]
    reference_rejected = reference_logps[1]

    # DPO preference logit:
    # beta * [(policy chosen - policy rejected)
    #         - (reference chosen - reference rejected)]
    preference_logit = beta * (
        (policy_chosen - policy_rejected)
        - (reference_chosen - reference_rejected)
    )

    loss = -F.logsigmoid(preference_logit)

    with torch.no_grad():
        chosen_reward = beta * (
            policy_chosen.detach() - reference_chosen
        )
        rejected_reward = beta * (
            policy_rejected.detach() - reference_rejected
        )

        reward_margin = chosen_reward - rejected_reward
        reward_accuracy = (reward_margin > 0).float()

    return loss, reward_accuracy, reward_margin


# -----------------------------
# 5. Validation
# -----------------------------

def score_dpo_dataset(
    policy,
    reference,
    tokenizer,
    rows,
    max_length=MAX_LENGTH,
    beta=BETA,
    description="Validation",
):
    """
    Calculate mean DPO loss, reward accuracy and reward margin.
    No gradients or optimizer updates are performed.
    """

    if len(rows) == 0:
        raise ValueError(f"{description} dataset is empty.")

    policy.eval()
    reference.eval()

    losses = []
    accuracies = []
    margins = []

    with torch.no_grad():
        for row in rows:
            batch = make_dpo_pair_batch(
                tokenizer, row, max_length
            )

            policy_batch = {
                key: value.to(POLICY_DEVICE)
                for key, value in batch.items()
            }

            reference_batch = {
                key: value.to(REFERENCE_DEVICE)
                for key, value in batch.items()
            }

            policy_logps = get_pair_logps(
                policy, policy_batch
            )

            reference_logps = get_pair_logps(
                reference, reference_batch
            ).to(POLICY_DEVICE)

            loss, accuracy, margin = dpo_pair_metrics(
                policy_logps, reference_logps, beta
            )

            losses.append(loss.item())
            accuracies.append(accuracy.item())
            margins.append(margin.item())

            del (
                batch,
                policy_batch,
                reference_batch,
                policy_logps,
                reference_logps,
                loss,
                accuracy,
                margin,
            )

    metrics = {
        "loss": sum(losses) / len(losses),
        "reward_accuracy": sum(accuracies) / len(accuracies),
        "reward_margin": sum(margins) / len(margins),
    }

    print(
        f"{description}: "
        f"loss={metrics['loss']:.4f}, "
        f"reward_accuracy={metrics['reward_accuracy']:.4f}, "
        f"reward_margin={metrics['reward_margin']:.4f}"
    )

    gc.collect()
    torch.cuda.empty_cache()

    return metrics


# -----------------------------
# 6. Save checkpoint and metrics
# -----------------------------

def save_dpo_checkpoint(model, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        output_dir,
        safe_serialization=True,
    )


# -----------------------------
# 7. DPO training
# -----------------------------

def train_dpo(
    policy,
    reference,
    tokenizer,
    train_rows,
    validation_rows,
    output_dir,
    epochs=1,
    max_train_examples=None,
    max_validation_examples=None,
    seed=SEED,
):
    """
    Train a policy with a frozen SFT reference model.

    Uses:
      - DPO loss
      - Adam-free RMSprop optimizer
      - Effective batch size of 32 preference pairs
      - Gradient accumulation
      - Gradient clipping
      - Gradient checkpointing
      - Best checkpoint selected by validation reward accuracy
    """

    train_rows = list(train_rows)
    validation_rows = list(validation_rows)

    if max_train_examples is not None:
        train_rows = train_rows[:max_train_examples]

    if max_validation_examples is not None:
        validation_rows = validation_rows[:max_validation_examples]

    if not train_rows:
        raise ValueError("Training dataset is empty.")
    if not validation_rows:
        raise ValueError("Validation dataset is empty.")

    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    best_dir = output_dir / "best_checkpoint"
    last_dir = output_dir / "last_checkpoint"

    # Ensure the reference remains frozen and in evaluation mode.
    reference.eval()
    for parameter in reference.parameters():
        parameter.requires_grad_(False)

    # Memory-saving settings for the trainable policy.
    policy.config.use_cache = False
    policy.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    optimizer = torch.optim.RMSprop(
        policy.parameters(),
        lr=LEARNING_RATE,
    )

    # Establish a baseline before any DPO updates.
    initial_metrics = score_dpo_dataset(
        policy=policy,
        reference=reference,
        tokenizer=tokenizer,
        rows=validation_rows,
        max_length=MAX_LENGTH,
        beta=BETA,
        description="Initial validation",
    )

    best_accuracy = initial_metrics["reward_accuracy"]
    best_loss = initial_metrics["loss"]

    # Save the baseline as the current best model.
    save_dpo_checkpoint(policy, best_dir)

    history = [{
        "epoch": 0,
        "step": 0,
        "split": "validation",
        **initial_metrics,
    }]

    global_step = 0

    steps_per_epoch = (
        len(train_rows) + GRAD_ACCUM_STEPS - 1
    ) // GRAD_ACCUM_STEPS

    print(
        f"Training pairs: {len(train_rows)} | "
        f"Validation pairs: {len(validation_rows)} | "
        f"Optimizer steps per epoch: {steps_per_epoch}"
    )

    for epoch in range(epochs):
        policy.train()

        indices = list(range(len(train_rows)))
        random.shuffle(indices)

        epoch_loss_sum = 0.0
        epoch_pair_count = 0

        # One optimizer update after each accumulation group.
        for group_start in range(
            0, len(indices), GRAD_ACCUM_STEPS
        ):
            group_indices = indices[
                group_start:group_start + GRAD_ACCUM_STEPS
            ]
            group_size = len(group_indices)

            optimizer.zero_grad(set_to_none=True)

            group_loss_sum = 0.0

            for index in group_indices:
                row = train_rows[index]

                batch = make_dpo_pair_batch(
                    tokenizer, row, MAX_LENGTH
                )

                policy_batch = {
                    key: value.to(POLICY_DEVICE)
                    for key, value in batch.items()
                }

                reference_batch = {
                    key: value.to(REFERENCE_DEVICE)
                    for key, value in batch.items()
                }

                # Policy forward pass: gradients are required.
                policy_logps = get_pair_logps(
                    policy, policy_batch
                )

                # Reference forward pass: no gradients.
                with torch.no_grad():
                    reference_logps = get_pair_logps(
                        reference, reference_batch
                    ).to(POLICY_DEVICE)

                loss, _, _ = dpo_pair_metrics(
                    policy_logps,
                    reference_logps,
                    BETA,
                )

                # Normalize accumulated gradients by group size.
                (loss / group_size).backward()

                pair_loss = loss.detach().item()
                group_loss_sum += pair_loss
                epoch_loss_sum += pair_loss
                epoch_pair_count += 1

                del (
                    batch,
                    policy_batch,
                    reference_batch,
                    policy_logps,
                    reference_logps,
                    loss,
                )

            # Clip the accumulated gradient before updating.
            grad_norm = torch.nn.utils.clip_grad_norm_(
                policy.parameters(),
                MAX_GRAD_NORM,
            )

            optimizer.step()
            global_step += 1

            mean_group_loss = group_loss_sum / group_size

            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"Step {global_step}/{epochs * steps_per_epoch} | "
                f"Mean pair loss={mean_group_loss:.4f} | "
                f"Grad norm={float(grad_norm):.2f}"
            )

            del grad_norm
            gc.collect()
            torch.cuda.empty_cache()

        # Evaluate after each epoch.
        validation_metrics = score_dpo_dataset(
            policy=policy,
            reference=reference,
            tokenizer=tokenizer,
            rows=validation_rows,
            max_length=MAX_LENGTH,
            beta=BETA,
            description=f"Epoch {epoch + 1} validation",
        )

        history.append({
            "epoch": epoch + 1,
            "step": global_step,
            "split": "validation",
            **validation_metrics,
        })

        mean_epoch_loss = (
            epoch_loss_sum / max(epoch_pair_count, 1)
        )

        print(
            f"Epoch {epoch + 1} training mean pair loss: "
            f"{mean_epoch_loss:.4f}"
        )

        # Best checkpoint: highest validation reward accuracy;
        # if tied, choose the lower validation DPO loss.
        current_accuracy = validation_metrics["reward_accuracy"]
        current_loss = validation_metrics["loss"]

        is_better = (
            current_accuracy > best_accuracy
            or (
                current_accuracy == best_accuracy
                and current_loss < best_loss
            )
        )

        if is_better:
            best_accuracy = current_accuracy
            best_loss = current_loss

            save_dpo_checkpoint(policy, best_dir)

            print(
                "Saved new best checkpoint: "
                f"reward_accuracy={best_accuracy:.4f}, "
                f"loss={best_loss:.4f}"
            )

        # Save the latest epoch checkpoint too.
        save_dpo_checkpoint(policy, last_dir)

        with open(output_dir / "metrics.json", "w") as f:
            json.dump(history, f, indent=2)

    print("\nDPO training finished.")
    print("Best checkpoint:", best_dir)
    print("Latest checkpoint:", last_dir)
    print("Metrics:", output_dir / "metrics.json")

    return history


In [ ]:

with open(
    REPO / "data/dpo/hh_rlhf_train.json",
    "r",
    encoding="utf-8",
) as f:
    hh_train = json.load(f)

with open(
    REPO / "data/dpo/hh_rlhf_validation.json",
    "r",
    encoding="utf-8",
) as f:
    hh_validation = json.load(f)

# Ensure the policy is the original SFT model, not a previous test policy.
# The current policy was reloaded from SFT in the previous cell.
policy.eval()

hh_smoke_history = train_dpo(
    policy=policy,
    reference=reference,
    tokenizer=tokenizer,
    train_rows=hh_train,
    validation_rows=hh_validation,
    output_dir=REPO / "outputs/dpo_hh_smoke",
    epochs=1,
    max_train_examples=64,
    max_validation_examples=32,
)

print("\nHH-RLHF smoke test finished.")


In [ ]:

import gc
import sys
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = Path("/kaggle/working/alignment-assignment-check")

SFT_PATH = (
    "/kaggle/input/datasets/khadijawn/stf-backup/"
    "outputs/sft_run/best_checkpoint"
)

BASE_MODEL = "Qwen/Qwen2.5-0.5B"

# Release the smoke-test models before reloading.
for name in ("policy", "reference"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Import the assignment implementation.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import submission_adapter as adapter

# Load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Trainable policy: original best SFT checkpoint on GPU 0.
policy = AutoModelForCausalLM.from_pretrained(
    SFT_PATH,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
).to("cuda:0")

policy.config.use_cache = False

# Frozen reference: same original SFT checkpoint on GPU 1.
reference = AutoModelForCausalLM.from_pretrained(
    SFT_PATH,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to("cuda:1")

reference.eval()
for parameter in reference.parameters():
    parameter.requires_grad_(False)

print("Original SFT policy and reference loaded.")
print("Policy device:", next(policy.parameters()).device)
print("Policy dtype:", next(policy.parameters()).dtype)
print("Reference device:", next(reference.parameters()).device)
print("Reference dtype:", next(reference.parameters()).dtype)
print("Reference frozen:", all(
    not parameter.requires_grad
    for parameter in reference.parameters()
))


In [ ]:

import json

HH_TRAIN_PATH = REPO / "data/dpo/hh_rlhf_train.json"
HH_VAL_PATH = REPO / "data/dpo/hh_rlhf_validation.json"

with open(HH_TRAIN_PATH, "r", encoding="utf-8") as f:
    hh_train = json.load(f)

with open(HH_VAL_PATH, "r", encoding="utf-8") as f:
    hh_validation = json.load(f)

print("HH-RLHF training pairs:", len(hh_train))
print("HH-RLHF validation pairs:", len(hh_validation))

assert len(hh_train) == 15000, "Unexpected training set size."
assert len(hh_validation) == 1000, "Unexpected validation set size."

print("\nExample row keys:", list(hh_train[0].keys()))
print("Dataset verification passed.")


In [ ]:

hh_full_history = train_dpo(
    policy=policy,
    reference=reference,
    tokenizer=tokenizer,
    train_rows=hh_train,
    validation_rows=hh_validation,
    output_dir=REPO / "outputs/dpo_hh_full",
    epochs=1,
    max_train_examples=None,
    max_validation_examples=None,
    seed=42,
)

print("\nFull HH-RLHF DPO training completed.")
print("Output directory:", REPO / "outputs/dpo_hh_full")


In [ ]:

from pathlib import Path
import shutil

INPUT = Path("/kaggle/input/datasets/khadijawn")
SOURCE = INPUT / "alignment-asssignment-recovery"
REPO = Path("/kaggle/working/alignment-assignment-check")

assert SOURCE.exists(), f"Recovery dataset not found: {SOURCE}"
assert (SOURCE / "submission_adapter.py").exists(), (
    "submission_adapter.py is missing from the recovery dataset."
)

# Restore a writable copy in the new session.
if REPO.exists():
    shutil.rmtree(REPO)

shutil.copytree(SOURCE, REPO)

print("Repository restored:", REPO.exists())
print("Adapter restored:", (REPO / "submission_adapter.py").exists())

print("\nFiles restored from the recovery dataset:")
for path in sorted(REPO.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO))


In [ ]:

REQUIRED_DATA = [
    "hh_rlhf_train.json",
    "hh_rlhf_validation.json",
    "ultrafeedback_train.json",
    "ultrafeedback_validation.json",
]

print("Checking restored repository for DPO data:\n")

for filename in REQUIRED_DATA:
    matches = list(REPO.rglob(filename))
    if matches:
        for path in matches:
            print(f"FOUND: {path}")
    else:
        print(f"MISSING: {filename}")


In [ ]:

from datasets import load_dataset

print("Loading HH-RLHF...")
hh = load_dataset("Anthropic/hh-rlhf", split="train")

print("\nHH-RLHF columns:", hh.column_names)
print("HH-RLHF rows:", len(hh))
print("\nHH-RLHF example:")
print("Chosen:", repr(hh[0]["chosen"][:500]))
print("Rejected:", repr(hh[0]["rejected"][:500]))

print("\n" + "=" * 60)
print("Loading UltraFeedback...")
uf = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)

print("\nUltraFeedback columns:", uf.column_names)
print("UltraFeedback rows:", len(uf))
print("\nUltraFeedback example:")
for key in uf.column_names:
    value = uf[0][key]
    print(f"\n{key} ({type(value).__name__}):")
    print(repr(value)[:1000])


In [ ]:

import json
import random
from pathlib import Path

# Use the restored repository.
REPO = Path("/kaggle/working/alignment-assignment-check")
DATA_DIR = REPO / "data" / "dpo"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_SIZE = 15_000
VALIDATION_SIZE = 1_000
ASSISTANT_MARKER = "\n\nAssistant:"


def convert_hh(row):
    """Split HH-RLHF conversations into prompt and preferred responses."""
    chosen = row.get("chosen", "")
    rejected = row.get("rejected", "")

    if not isinstance(chosen, str) or not isinstance(rejected, str):
        return None

    # Find the longest shared prefix of the two conversations.
    common_length = 0
    for a, b in zip(chosen, rejected):
        if a != b:
            break
        common_length += 1

    common_prefix = chosen[:common_length]

    # Find the last assistant-turn marker in the shared prefix.
    marker_pos = common_prefix.rfind(ASSISTANT_MARKER)
    if marker_pos < 0:
        return None

    # Keep the assistant marker in the prompt.
    prompt_end = marker_pos + len(ASSISTANT_MARKER)
    prompt = chosen[:prompt_end].strip()
    chosen_response = chosen[prompt_end:].strip()
    rejected_response = rejected[prompt_end:].strip()

    if not prompt or not chosen_response or not rejected_response:
        return None

    if chosen_response == rejected_response:
        return None

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
    }


def last_assistant_message(messages):
    """Extract the final assistant response from a message list."""
    if not isinstance(messages, list):
        return ""

    for message in reversed(messages):
        if (
            isinstance(message, dict)
            and message.get("role") == "assistant"
        ):
            content = message.get("content", "")
            if isinstance(content, str):
                return content.strip()

    return ""


def convert_ultrafeedback(row):
    """Convert UltraFeedback preference records."""
    prompt_text = row.get("prompt", "")
    chosen_response = last_assistant_message(row.get("chosen"))
    rejected_response = last_assistant_message(row.get("rejected"))

    if not isinstance(prompt_text, str):
        return None

    prompt_text = prompt_text.strip()

    if not prompt_text or not chosen_response or not rejected_response:
        return None

    if chosen_response == rejected_response:
        return None

    # Match the Human/Assistant prompt convention used by HH-RLHF.
    prompt = f"\n\nHuman: {prompt_text}\n\nAssistant:"

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
    }


def make_splits(dataset, converter, name):
    converted = []
    skipped = 0

    for row in dataset:
        example = converter(row)
        if example is None:
            skipped += 1
        else:
            converted.append(example)

    # Deterministic shuffle so the same seed reproduces the same split.
    rng = random.Random(SEED)
    rng.shuffle(converted)

    required = TRAIN_SIZE + VALIDATION_SIZE
    if len(converted) < required:
        raise ValueError(
            f"{name}: only {len(converted)} valid pairs; "
            f"need at least {required}."
        )

    validation = converted[:VALIDATION_SIZE]
    train = converted[VALIDATION_SIZE:required]

    train_path = DATA_DIR / f"{name}_train.json"
    val_path = DATA_DIR / f"{name}_validation.json"

    for path, rows in [(train_path, train), (val_path, validation)]:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(rows, f, ensure_ascii=False)

    print(f"\n{name.upper()}")
    print("Valid pairs:", len(converted))
    print("Skipped pairs:", skipped)
    print("Training pairs:", len(train))
    print("Validation pairs:", len(validation))
    print("Training file:", train_path)
    print("Validation file:", val_path)

    # Verify the files can be read and have the required fields.
    for path, expected_count in [
        (train_path, TRAIN_SIZE),
        (val_path, VALIDATION_SIZE),
    ]:
        with open(path, "r", encoding="utf-8") as f:
            saved = json.load(f)

        assert len(saved) == expected_count
        assert all(
            set(("prompt", "chosen", "rejected")).issubset(row)
            and all(
                isinstance(row[key], str) and row[key].strip()
                for key in ("prompt", "chosen", "rejected")
            )
            for row in saved
        )

    print("File validation: PASSED")


# These are the datasets loaded in the previous cell.
make_splits(hh, convert_hh, "hh_rlhf")
make_splits(uf, convert_ultrafeedback, "ultrafeedback")

print("\nAll four DPO dataset files have been created and checked.")


In [ ]:

from pathlib import Path
import sys
import inspect
import torch
import shutil
import json

REPO = Path("/kaggle/working/alignment-assignment-check")
DATA_DIR = REPO / "data" / "dpo"
SFT_CHECKPOINT = Path(
    "/kaggle/input/datasets/khadijawn/"
    "stf-backup/outputs/sft_run/best_checkpoint"
)

# Verify the repository and adapter file first.
assert REPO.exists(), f"Repository not found: {REPO}"
assert (REPO / "submission_adapter.py").exists(), (
    f"Adapter file not found: {REPO / 'submission_adapter.py'}"
)

# Make the adapter importable from this notebook.
sys.path.insert(0, str(REPO))
import submission_adapter

print("=" * 55)
print("1. GPU STATUS")
print("=" * 55)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {props.name} | "
        f"Total: {total / 1024**3:.2f} GiB | "
        f"Free: {free / 1024**3:.2f} GiB"
    )

print("\n" + "=" * 55)
print("2. DISK SPACE")
print("=" * 55)
usage = shutil.disk_usage("/kaggle/working")
print(f"Total: {usage.total / 1024**3:.2f} GiB")
print(f"Free: {usage.free / 1024**3:.2f} GiB")

print("\n" + "=" * 55)
print("3. SFT CHECKPOINT")
print("=" * 55)
assert (SFT_CHECKPOINT / "config.json").exists()
assert (SFT_CHECKPOINT / "model.safetensors").exists()
print("Original best SFT checkpoint: available")
print(
    "Model size:",
    f"{(SFT_CHECKPOINT / 'model.safetensors').stat().st_size / 1024**3:.2f} GiB"
)

print("\n" + "=" * 55)
print("4. DPO DATA FILES")
print("=" * 55)
for filename in [
    "hh_rlhf_train.json",
    "hh_rlhf_validation.json",
    "ultrafeedback_train.json",
    "ultrafeedback_validation.json",
]:
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"
    print(f"{filename}: {path.stat().st_size / 1024**2:.2f} MiB")

print("\n" + "=" * 55)
print("5. ADAPTER FUNCTIONS")
print("=" * 55)
for name, obj in inspect.getmembers(
    submission_adapter, inspect.isfunction
):
    if name in {
        "make_dpo_pair_batch",
        "get_pair_logps",
        "dpo_pair_metrics",
        "score_dpo_dataset",
        "train_dpo",
    }:
        print(f"\n{name}{inspect.signature(obj)}")
        print(inspect.getsource(obj)[:1800])

print("\nEnvironment check complete. No training was started.")


In [ ]:

import inspect
import submission_adapter

print("Adapter file:", submission_adapter.__file__)

print("\nFunctions defined in the adapter:")
for name, obj in inspect.getmembers(
    submission_adapter, inspect.isfunction
):
    if obj.__module__ == submission_adapter.__name__:
        print(f"  {name}{inspect.signature(obj)}")

print("\n" + "=" * 65)
print("FULL ADAPTER SOURCE")
print("=" * 65)

with open(submission_adapter.__file__, "r", encoding="utf-8") as f:
    print(f.read())


In [ ]:

from pathlib import Path
import sys
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = Path("/kaggle/working/alignment-assignment-check")
sys.path.insert(0, str(REPO))

from submission_adapter import build_sft_batch

SFT_CHECKPOINT = Path(
    "/kaggle/input/datasets/khadijawn/"
    "stf-backup/outputs/sft_run/best_checkpoint"
)
HH_TRAIN = REPO / "data/dpo/hh_rlhf_train.json"

assert torch.cuda.is_available() and torch.cuda.device_count() >= 2
assert SFT_CHECKPOINT.exists() and HH_TRAIN.exists()

# Load one real preference pair.
with open(HH_TRAIN, "r", encoding="utf-8") as f:
    example = json.load(f)[0]

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Policy starts from the original best SFT checkpoint, in FP32.
print("Loading trainable policy on GPU 0...")
policy = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
).to("cuda:0")

# Reference is a separate, frozen copy of the same SFT checkpoint.
print("Loading frozen reference on GPU 1...")
reference = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to("cuda:1")

policy.config.use_cache = False
reference.config.use_cache = False
reference.eval()

for parameter in reference.parameters():
    parameter.requires_grad_(False)

# Save activation memory during policy backpropagation.
policy.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
policy.train()


def chunked_response_logps(model, batch, chunk_size=4):
    """
    Sum response-token log probabilities without creating full-sequence
    vocabulary logits. Only small chunks go through the LM head.
    """
    device = next(model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items()}

    # Qwen2.5's transformer backbone returns final hidden states.
    outputs = model.model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        use_cache=False,
    )
    hidden = outputs.last_hidden_state

    # Hidden state at t-1 predicts token at t.
    targets = batch["labels"][:, 1:]
    valid = targets.ne(-100)
    selected_hidden = hidden[:, :-1, :][valid]
    selected_targets = targets[valid]

    if selected_targets.numel() == 0:
        raise ValueError("This example contains no response tokens.")

    total_logp = torch.zeros((), device=device, dtype=torch.float32)

    for start in range(0, selected_targets.numel(), chunk_size):
        end = min(start + chunk_size, selected_targets.numel())

        # Chunked vocabulary projection reduces peak memory.
        logits = model.lm_head(selected_hidden[start:end]).float()
        target_ids = selected_targets[start:end]

        target_logits = logits.gather(
            1, target_ids.unsqueeze(1)
        ).squeeze(1)

        token_logps = target_logits - torch.logsumexp(logits, dim=1)
        total_logp = total_logp + token_logps.sum()

    return total_logp


def one_response_batch(response):
    return build_sft_batch(
        tokenizer,
        [example["prompt"]],
        [response],
        max_length=512,
    )


print("\nComputing policy log probabilities...")
policy_chosen = chunked_response_logps(
    policy, one_response_batch(example["chosen"])
)
policy_rejected = chunked_response_logps(
    policy, one_response_batch(example["rejected"])
)

print("Computing reference log probabilities...")
with torch.no_grad():
    reference_chosen = chunked_response_logps(
        reference, one_response_batch(example["chosen"])
    )
    reference_rejected = chunked_response_logps(
        reference, one_response_batch(example["rejected"])
    )

beta = 0.1
preference_logit = beta * (
    (policy_chosen - policy_rejected)
    - (reference_chosen - reference_rejected)
)
loss = -F.logsigmoid(preference_logit)

print("\nOne-pair DPO smoke test:")
print("Policy chosen logp:", policy_chosen.item())
print("Policy rejected logp:", policy_rejected.item())
print("Reference chosen logp:", reference_chosen.item())
print("Reference rejected logp:", reference_rejected.item())
print("DPO loss:", loss.item())

print("\nTesting backward pass (no optimizer step)...")
loss.backward()

grad = policy.model.layers[0].self_attn.q_proj.weight.grad
assert grad is not None, "No policy gradient was produced."
assert torch.isfinite(grad).all(), "Policy gradient contains non-finite values."

print("Policy gradient exists and is finite:", True)

# Do not change model weights in this test.
policy.zero_grad(set_to_none=True)
del loss, policy_chosen, policy_rejected
del reference_chosen, reference_rejected, preference_logit
torch.cuda.empty_cache()

print("\nSMOKE TEST PASSED — no training update was performed.")


In [ ]:

import torch
import torch.nn.functional as F

beta = 0.1

# Move the reference's scalar log-probabilities to the policy GPU.
reference_chosen_on_policy = reference_chosen.to(policy_chosen.device)
reference_rejected_on_policy = reference_rejected.to(policy_chosen.device)

preference_logit = beta * (
    (policy_chosen - policy_rejected)
    - (
        reference_chosen_on_policy
        - reference_rejected_on_policy
    )
)

loss = -F.logsigmoid(preference_logit)

print("One-pair DPO smoke test:")
print("Policy chosen logp:", policy_chosen.item())
print("Policy rejected logp:", policy_rejected.item())
print("Reference chosen logp:", reference_chosen.item())
print("Reference rejected logp:", reference_rejected.item())
print("DPO loss:", loss.item())

print("\nTesting backward pass (no optimizer step)...")
loss.backward()

grad = policy.model.layers[0].self_attn.q_proj.weight.grad

assert grad is not None, "No policy gradient was produced."
assert torch.isfinite(grad).all(), "Policy gradient contains non-finite values."

print("Policy gradient exists and is finite:", True)

# No optimizer step: model weights remain unchanged.
policy.zero_grad(set_to_none=True)
torch.cuda.empty_cache()

print("\nSMOKE TEST PASSED — no training update was performed.")


In [ ]:

import os
import json
import random
import shutil
from pathlib import Path

import torch
import torch.nn.functional as F
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
REPO = Path("/kaggle/working/alignment-assignment-check")
DATA_DIR = REPO / "data" / "dpo"
OUTPUT_DIR = REPO / "outputs" / "dpo_hh_full"

LAST_DIR = OUTPUT_DIR / "last_checkpoint"
BEST_DIR = OUTPUT_DIR / "best_checkpoint"

MAX_LENGTH = 512
BETA = 0.1
LEARNING_RATE = 1e-6
GRAD_ACCUM_STEPS = 32
MAX_GRAD_NORM = 1.0
SAVE_EVERY_STEPS = 10
EVAL_EVERY_STEPS = 100
EVAL_SUBSET_SIZE = 100

assert "policy" in globals(), "Run the smoke-test model-loading cell first."
assert "reference" in globals(), "Reference model is not loaded."
assert "tokenizer" in globals(), "Tokenizer is not loaded."
assert "chunked_response_logps" in globals(), "Chunked log-probability function is missing."

assert torch.cuda.is_available()
assert next(policy.parameters()).device.type == "cuda"
assert next(reference.parameters()).device.type == "cuda"

print("Policy device:", next(policy.parameters()).device)
print("Reference device:", next(reference.parameters()).device)
print("Output directory:", OUTPUT_DIR)


# ---------------------------------------------------------
# Data helpers
# ---------------------------------------------------------
def load_dpo_json(filename):
    path = DATA_DIR / filename
    with open(path, "r", encoding="utf-8") as f:
        rows = json.load(f)

    required = {"prompt", "chosen", "rejected"}
    assert rows and all(required.issubset(row) for row in rows)
    return rows


def response_batch(prompt, response):
    return {
        key: value
        for key, value in __import__(
            "submission_adapter"
        ).build_sft_batch(
            tokenizer,
            [prompt],
            [response],
            MAX_LENGTH,
        ).items()
    }


def calculate_pair(policy_model, reference_model, row):
    """Compute one pair's DPO loss and reward margin."""
    chosen_batch = response_batch(row["prompt"], row["chosen"])
    rejected_batch = response_batch(row["prompt"], row["rejected"])

    policy_chosen = chunked_response_logps(policy_model, chosen_batch)
    policy_rejected = chunked_response_logps(policy_model, rejected_batch)

    with torch.no_grad():
        reference_chosen = chunked_response_logps(
            reference_model, chosen_batch
        )
        reference_rejected = chunked_response_logps(
            reference_model, rejected_batch
        )

    # The two models live on different GPUs, so move only the scalar
    # reference results to the policy GPU.
    device = policy_chosen.device
    reference_chosen = reference_chosen.to(device)
    reference_rejected = reference_rejected.to(device)

    margin = BETA * (
        (policy_chosen - policy_rejected)
        - (reference_chosen - reference_rejected)
    )
    loss = -F.logsigmoid(margin)

    return loss, margin.detach()


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------
def evaluate_dpo(policy_model, reference_model, rows, label):
    was_training = policy_model.training
    policy_model.eval()
    reference_model.eval()

    losses = []
    margins = []

    with torch.no_grad():
        for i, row in enumerate(rows):
            loss, margin = calculate_pair(
                policy_model, reference_model, row
            )
            losses.append(float(loss.item()))
            margins.append(float(margin.item()))

            if (i + 1) % 50 == 0:
                print(f"  {label}: evaluated {i + 1}/{len(rows)} pairs")

    if was_training:
        policy_model.train()

    metrics = {
        "validation_dpo_loss": sum(losses) / len(losses),
        "validation_reward_accuracy": sum(
            margin > 0 for margin in margins
        ) / len(margins),
        "validation_reward_margin": sum(margins) / len(margins),
        "num_pairs": len(rows),
    }

    print(
        f"{label}: "
        f"loss={metrics['validation_dpo_loss']:.4f}, "
        f"reward_accuracy={metrics['validation_reward_accuracy']:.4f}, "
        f"reward_margin={metrics['validation_reward_margin']:.4f}"
    )
    return metrics


# ---------------------------------------------------------
# Checkpoint helpers
# ---------------------------------------------------------
def capture_rng_state():
    return {
        "python_rng": random.getstate(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
    }


def restore_rng_state(state):
    random.setstate(state["python_rng"])
    torch.set_rng_state(state["torch_rng"])
    torch.cuda.set_rng_state_all(state["cuda_rng"])


def save_latest_checkpoint(model, optimizer, state):
    """
    Save model + optimizer + exact data position into a temporary
    directory, then replace the previous rolling checkpoint.
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    temp_dir = OUTPUT_DIR / "last_checkpoint.tmp"
    backup_dir = OUTPUT_DIR / "last_checkpoint.old"

    shutil.rmtree(temp_dir, ignore_errors=True)
    shutil.rmtree(backup_dir, ignore_errors=True)
    temp_dir.mkdir(parents=True)

    model.save_pretrained(temp_dir, safe_serialization=True)
    torch.save(optimizer.state_dict(), temp_dir / "optimizer.pt")
    torch.save(state, temp_dir / "trainer_state.pt")

    with open(temp_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(state["history"], f, indent=2)

    # Keep the previous valid checkpoint until the new one is complete.
    if LAST_DIR.exists():
        os.replace(LAST_DIR, backup_dir)

    try:
        os.replace(temp_dir, LAST_DIR)
    except Exception:
        # If replacement fails, preserve the previous checkpoint.
        if not LAST_DIR.exists() and backup_dir.exists():
            os.replace(backup_dir, LAST_DIR)
        raise

    shutil.rmtree(backup_dir, ignore_errors=True)
    print(
        f"Saved recovery checkpoint at optimizer step "
        f"{state['global_step']}."
    )


def load_latest_checkpoint(model, optimizer):
    """Restore model weights, optimizer, data position and RNG."""
    checkpoint_dir = LAST_DIR

    # Recover the previous complete copy if an interruption happened
    # during the directory replacement.
    backup_dir = OUTPUT_DIR / "last_checkpoint.old"
    if not checkpoint_dir.exists() and backup_dir.exists():
        os.replace(backup_dir, checkpoint_dir)

    state_path = checkpoint_dir / "trainer_state.pt"
    model_path = checkpoint_dir / "model.safetensors"
    optimizer_path = checkpoint_dir / "optimizer.pt"

    if not (
        state_path.exists()
        and model_path.exists()
        and optimizer_path.exists()
    ):
        raise FileNotFoundError(
            f"No complete recovery checkpoint found in {checkpoint_dir}"
        )

    print("Loading model weights from recovery checkpoint...")
    weights = load_file(str(model_path), device="cpu")
    model.load_state_dict(weights)
    del weights

    state = torch.load(
        state_path, map_location="cpu", weights_only=False
    )
    optimizer_state = torch.load(
        optimizer_path, map_location="cpu", weights_only=False
    )
    optimizer.load_state_dict(optimizer_state)

    restore_rng_state(state["rng_state"])
    print(
        "Resumed at optimizer step", state["global_step"],
        "| examples already processed:", state["next_index"]
    )
    return state


def save_best_checkpoint(model, metrics):
    """Save the best policy weights based on full validation accuracy."""
    temp_dir = OUTPUT_DIR / "best_checkpoint.tmp"
    backup_dir = OUTPUT_DIR / "best_checkpoint.old"

    shutil.rmtree(temp_dir, ignore_errors=True)
    shutil.rmtree(backup_dir, ignore_errors=True)
    temp_dir.mkdir(parents=True)

    model.save_pretrained(temp_dir, safe_serialization=True)
    with open(temp_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    if BEST_DIR.exists():
        os.replace(BEST_DIR, backup_dir)

    try:
        os.replace(temp_dir, BEST_DIR)
    except Exception:
        if not BEST_DIR.exists() and backup_dir.exists():
            os.replace(backup_dir, BEST_DIR)
        raise

    shutil.rmtree(backup_dir, ignore_errors=True)
    print("Saved best model:", BEST_DIR)


# ---------------------------------------------------------
# Resumable training
# ---------------------------------------------------------
def train_dpo_resumable(
    train_rows,
    validation_rows,
    resume=False,
    seed=42,
):
    """
    Train one DPO epoch.

    Resume only from a checkpoint made by this function.
    If resume=False, the supplied policy must be the original SFT model.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    optimizer = torch.optim.RMSprop(
        policy.parameters(),
        lr=LEARNING_RATE,
    )

    if resume:
        state = load_latest_checkpoint(policy, optimizer)
        indices = state["indices"]
    else:
        if LAST_DIR.exists():
            raise FileExistsError(
                f"{LAST_DIR} already exists. Use resume=True to continue, "
                "or choose a new OUTPUT_DIR for a fresh experiment."
            )

        indices = list(range(len(train_rows)))
        random.Random(seed).shuffle(indices)

        state = {
            "epoch": 0,
            "global_step": 0,
            "next_index": 0,
            "indices": indices,
            "best_validation_accuracy": -1.0,
            "history": [],
            "rng_state": capture_rng_state(),
        }

        # Initial recovery point before any parameter updates.
        save_latest_checkpoint(policy, optimizer, state)

    policy.train()
    reference.eval()

    total_examples = len(indices)
    next_index = state["next_index"]
    global_step = state["global_step"]

    while next_index < total_examples:
        group_indices = indices[
            next_index : next_index + GRAD_ACCUM_STEPS
        ]
        group_size = len(group_indices)

        optimizer.zero_grad(set_to_none=True)
        pair_losses = []

        for index in group_indices:
            row = train_rows[index]
            loss, _ = calculate_pair(policy, reference, row)

            # Average gradients over the actual group size, including
            # the smaller final group if the dataset size isn't divisible.
            (loss / group_size).backward()
            pair_losses.append(float(loss.detach().item()))

            del loss

        grad_norm = torch.nn.utils.clip_grad_norm_(
            policy.parameters(), MAX_GRAD_NORM
        )
        optimizer.step()

        next_index += group_size
        global_step += 1

        step_record = {
            "step": global_step,
            "examples_processed": next_index,
            "training_pair_loss": sum(pair_losses) / len(pair_losses),
            "gradient_norm": float(grad_norm.item()),
        }

        print(
            f"Step {global_step} | "
            f"Pairs {next_index}/{total_examples} | "
            f"Train loss={step_record['training_pair_loss']:.4f} | "
            f"Grad norm={step_record['gradient_norm']:.2f}"
        )

        state.update({
            "epoch": 0,
            "global_step": global_step,
            "next_index": next_index,
            "indices": indices,
            "last_step": step_record,
            "rng_state": capture_rng_state(),
        })

        # Periodic monitoring on a fixed, small validation subset.
        if (
            global_step % EVAL_EVERY_STEPS == 0
            and next_index < total_examples
        ):
            subset = validation_rows[:min(
                EVAL_SUBSET_SIZE, len(validation_rows)
            )]
            subset_metrics = evaluate_dpo(
                policy, reference, subset,
                label=f"Step {global_step} validation subset"
            )
            state["history"].append({
                "step": global_step,
                "type": "validation_subset",
                **subset_metrics,
            })
            state["rng_state"] = capture_rng_state()

        # Save the latest state at each configured interval and at the end.
        if (
            global_step % SAVE_EVERY_STEPS == 0
            or next_index >= total_examples
        ):
            save_latest_checkpoint(policy, optimizer, state)

    # Final metrics use the complete validation set.
    full_metrics = evaluate_dpo(
        policy, reference, validation_rows,
        label="Full validation"
    )
    full_metrics["global_step"] = global_step
    state["history"].append({
        "step": global_step,
        "type": "full_validation",
        **full_metrics,
    })

    # Choose the best checkpoint using full validation reward accuracy.
    best_accuracy = state["best_validation_accuracy"]
    if full_metrics["validation_reward_accuracy"] >= best_accuracy:
        save_best_checkpoint(policy, full_metrics)
        state["best_validation_accuracy"] = (
            full_metrics["validation_reward_accuracy"]
        )

    state["rng_state"] = capture_rng_state()
    save_latest_checkpoint(policy, optimizer, state)

    with open(OUTPUT_DIR / "final_metrics.json", "w", encoding="utf-8") as f:
        json.dump(state["history"], f, indent=2)

    print("\nDPO training run finished.")
    print("Latest resumable checkpoint:", LAST_DIR)
    print("Best checkpoint:", BEST_DIR)
    print("Final metrics:", OUTPUT_DIR / "final_metrics.json")

    return full_metrics


print("\nTrainer functions defined successfully.")
print("No training has started yet.")


In [ ]:

from pathlib import Path
import json
import torch

REPO = Path("/kaggle/working/alignment-assignment-check")
DATA_DIR = REPO / "data" / "dpo"

with open(DATA_DIR / "hh_rlhf_train.json", "r", encoding="utf-8") as f:
    hh_train = json.load(f)

with open(DATA_DIR / "hh_rlhf_validation.json", "r", encoding="utf-8") as f:
    hh_validation = json.load(f)

assert len(hh_train) == 15000
assert len(hh_validation) == 1000
assert next(policy.parameters()).device == torch.device("cuda:0")
assert next(reference.parameters()).device == torch.device("cuda:1")

print("HH-RLHF training pairs:", len(hh_train))
print("HH-RLHF validation pairs:", len(hh_validation))
print("Policy and reference are on the expected GPUs.")
print("Ready to launch when you confirm.")

from pathlib import Path
import json
import torch

REPO = Path("/kaggle/working/alignment-assignment-check")
DATA_DIR = REPO / "data" / "dpo"

with open(DATA_DIR / "hh_rlhf_train.json", "r", encoding="utf-8") as f:
    hh_train = json.load(f)

with open(DATA_DIR / "hh_rlhf_validation.json", "r", encoding="utf-8") as f:
    hh_validation = json.load(f)

assert len(hh_train) == 15000
assert len(hh_validation) == 1000
assert next(policy.parameters()).device == torch.device("cuda:0")
assert next(reference.parameters()).device == torch.device("cuda:1")

print("HH-RLHF training pairs:", len(hh_train))
print("HH-RLHF validation pairs:", len(hh_validation))
print("Policy and reference are on the expected GPUs.")
print("Ready to launch when you confirm.")


In [ ]:
from pathlib import Path
import json
import os

LAST_DIR = Path(
    "/kaggle/working/alignment-assignment-check/outputs/dpo_hh_full/last_checkpoint"
)

print("Checkpoint exists:", LAST_DIR.exists())
print("\nFiles in checkpoint:")

if LAST_DIR.exists():
    for item in sorted(LAST_DIR.iterdir()):
        if item.is_file():
            print(f"  {item.name} — {item.stat().st_size / (1024**2):.2f} MB")
        else:
            print(f"  {item.name}/")

    state_file = LAST_DIR / "trainer_state.pt"
    model_file = LAST_DIR / "model.safetensors"

    print("\nTrainer state exists:", state_file.exists())
    print("Model weights exist:", model_file.exists())

In [ ]:

from safetensors import safe_open
import torch

checkpoint = LAST_DIR / "model.safetensors"

with safe_open(str(checkpoint), framework="pt", device="cpu") as f:
    checkpoint_keys = set(f.keys())

print("Model config ties input/output embeddings:",
      policy.config.tie_word_embeddings)

print("Input embedding shape:",
      tuple(policy.model.embed_tokens.weight.shape))

print("Output layer shape:",
      tuple(policy.lm_head.weight.shape))

print("Input/output weights share memory:",
      policy.model.embed_tokens.weight.data_ptr()
      == policy.lm_head.weight.data_ptr())

print("lm_head.weight saved in checkpoint:",
      "lm_head.weight" in checkpoint_keys)

print("Embedding weights saved in checkpoint:",
      "model.embed_tokens.weight" in checkpoint_keys)

trainer_state = torch.load(
    LAST_DIR / "trainer_state.pt",
    map_location="cpu",
    weights_only=False,
)

print("\nSaved global step:", trainer_state.get("global_step"))
print("Saved next training index:", trainer_state.get("next_index"))


In [ ]:

def load_latest_checkpoint(model, optimizer):
    import torch
    from safetensors.torch import load_file

    checkpoint_dir = LAST_DIR

    if not checkpoint_dir.exists() and (OUTPUT_DIR / "last_checkpoint.old").exists():
        checkpoint_dir = OUTPUT_DIR / "last_checkpoint.old"

    model_path = checkpoint_dir / "model.safetensors"
    optimizer_path = checkpoint_dir / "optimizer.pt"
    state_path = checkpoint_dir / "trainer_state.pt"

    if not model_path.exists() or not state_path.exists():
        raise FileNotFoundError(
            f"Incomplete recovery checkpoint: {checkpoint_dir}"
        )

    print(f"Loading model weights from: {checkpoint_dir}")

    weights = load_file(str(model_path), device="cpu")

    # Qwen2.5 ties its output layer to its input embeddings.
    # Some safetensors checkpoints omit lm_head.weight because
    # it is the same tensor as model.embed_tokens.weight.
    if (
        model.config.tie_word_embeddings
        and "lm_head.weight" not in weights
        and "model.embed_tokens.weight" in weights
    ):
        weights["lm_head.weight"] = weights["model.embed_tokens.weight"]
        print("Restored missing lm_head.weight from shared embeddings.")

    incompatible = model.load_state_dict(weights, strict=True)
    del weights

    print("Model weights restored successfully.")

    if optimizer_path.exists():
        optimizer_state = torch.load(
            optimizer_path,
            map_location="cpu",
            weights_only=False,
        )
        optimizer.load_state_dict(optimizer_state)
        del optimizer_state
        print("Optimizer state restored.")
    else:
        print("No optimizer state file found; optimizer remains fresh.")

    state = torch.load(
        state_path,
        map_location="cpu",
        weights_only=False,
    )

    if "rng_state" in state:
        restore_rng_state(state["rng_state"])

    print(
        f"Resuming at global step {state['global_step']}, "
        f"training index {state['next_index']}."
    )

    return state


In [ ]:

# Disable activation recomputation on the trainable policy.
policy.gradient_checkpointing_disable()

# Keep the model settings suitable for training.
policy.config.use_cache = False
policy.train()

# Clear any partial gradients left by the failed backward pass.
policy.zero_grad(set_to_none=True)

print("Gradient checkpointing disabled.")
print("Policy device:", next(policy.parameters()).device)
print("Policy training mode:", policy.training)
print(
    "Gradient checkpointing enabled:",
    getattr(policy, "is_gradient_checkpointing", "unknown")
)


In [ ]:

hh_metrics = train_dpo_resumable(
    train_rows=hh_train,
    validation_rows=hh_validation,
    resume=True,
    seed=42,
)

print("\nFinal HH-RLHF metrics:")
print(json.dumps(hh_metrics, indent=2))


In [ ]:
from pathlib import Path
import shutil
import json

REPO = Path("/kaggle/working/alignment-assignment-check")

BEST_DIR = REPO / "outputs" / "dpo_hh_full" / "best_checkpoint"
METRICS_FILE = REPO / "outputs" / "dpo_hh_full" / "final_metrics.json"

BACKUP_DIR = Path("/kaggle/working/hh_rlhf_dpo_backup")
BACKUP_MODEL_DIR = BACKUP_DIR / "best_checkpoint"

# Check that the completed run exists
assert BEST_DIR.exists(), f"Best checkpoint not found: {BEST_DIR}"
assert (BEST_DIR / "model.safetensors").exists(), "Model weights are missing!"
assert METRICS_FILE.exists(), "Final metrics file is missing!"

# Copy the best model and metrics
if BACKUP_MODEL_DIR.exists():
    shutil.rmtree(BACKUP_MODEL_DIR)

shutil.copytree(BEST_DIR, BACKUP_MODEL_DIR)
shutil.copy2(METRICS_FILE, BACKUP_DIR / "final_metrics.json")

# Save a short summary too
with open(BACKUP_DIR / "README.txt", "w") as f:
    f.write("Completed from-scratch HH-RLHF DPO run\n")
    f.write("Base model: Qwen/Qwen2.5-0.5B\n")
    f.write("Best checkpoint: best_checkpoint/\n")
    f.write("Final metrics: final_metrics.json\n")

print("Backup folder:", BACKUP_DIR)
print("Backup contents:")
for path in sorted(BACKUP_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(BACKUP_DIR)} — {path.stat().st_size / (1024**2):.1f} MB")

In [ ]:

from pathlib import Path
import shutil

REPO = Path("/kaggle/working/alignment-assignment-check")
SOURCE_DIR = REPO / "outputs" / "dpo_hh_full" / "best_checkpoint"
METRICS_FILE = REPO / "outputs" / "dpo_hh_full" / "final_metrics.json"

BACKUP_DIR = Path("/kaggle/working/hh_rlhf_dpo_backup")

# Verify that the important files exist
assert (SOURCE_DIR / "model.safetensors").exists(), "Model weights not found!"
assert METRICS_FILE.exists(), "Final metrics not found!"

# Ensure the backup contains the latest best checkpoint
if (BACKUP_DIR / "best_checkpoint").exists():
    shutil.rmtree(BACKUP_DIR / "best_checkpoint")

shutil.copytree(SOURCE_DIR, BACKUP_DIR / "best_checkpoint")
shutil.copy2(METRICS_FILE, BACKUP_DIR / "final_metrics.json")

# Create the ZIP archive
ZIP_BASE = "/kaggle/working/hh_rlhf_dpo_backup"
ZIP_PATH = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir="/kaggle/working",
    base_dir="hh_rlhf_dpo_backup"
)

zip_file = Path(ZIP_PATH)

print("ZIP created:", zip_file)
print(f"ZIP size: {zip_file.stat().st_size / (1024**3):.2f} GB")
print("Contains model weights and final metrics.")


In [ ]:
from pathlib import Path

p = Path("/kaggle/working/hh_rlhf_dpo_backup.zip")
print("Exists:", p.exists())
if p.exists():
    print("Size:", round(p.stat().st_size / (1024**3), 2), "GB")

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

p = Path("/kaggle/working/hh_rlhf_dpo_backup.zip")
assert p.exists(), "ZIP file does not exist!"

display(FileLink(str(p)))

In [ ]:
import os
from pathlib import Path

print("Kaggle username:", os.environ.get("KAGGLE_USERNAME"))
print("Kaggle API token configured:", Path("/root/.kaggle/kaggle.json").exists())
print("Backup ZIP exists:", Path("/kaggle/working/hh_rlhf_dpo_backup.zip").exists())

In [31]:
import os
import zipfile

source = "/kaggle/working/hh_rlhf_dpo_backup.zip"
parts_dir = "/kaggle/working/hh_rlhf_dpo_backup_parts"

os.makedirs(parts_dir, exist_ok=True)

part_size = 400 * 1024 * 1024  # 400 MB

with open(source, "rb") as f:
    part_number = 1
    while True:
        chunk = f.read(part_size)
        if not chunk:
            break

        part_path = os.path.join(
            parts_dir, f"part_{part_number:02d}.bin"
        )
        with open(part_path, "wb") as part:
            part.write(chunk)

        print("Created:", part_path, round(len(chunk) / 1e6), "MB")
        part_number += 1

print("\nAll parts created:")
print(os.listdir(parts_dir))

Created: /kaggle/working/hh_rlhf_dpo_backup_parts/part_01.bin 419 MB
Created: /kaggle/working/hh_rlhf_dpo_backup_parts/part_02.bin 419 MB
Created: /kaggle/working/hh_rlhf_dpo_backup_parts/part_03.bin 419 MB
Created: /kaggle/working/hh_rlhf_dpo_backup_parts/part_04.bin 419 MB
Created: /kaggle/working/hh_rlhf_dpo_backup_parts/part_05.bin 156 MB

All parts created:
['part_02.bin', 'part_05.bin', 'part_01.bin', 'part_04.bin', 'part_03.bin']


In [32]:
import os

paths = {
    "Original SFT checkpoint": (
        "/kaggle/input/datasets/khadijawn/"
        "stf-backup/outputs/sft_run/best_checkpoint"
    ),
    "HH-RLHF best checkpoint": (
        "/kaggle/working/alignment-assignment-check/"
        "outputs/dpo_hh_full/best_checkpoint"
    ),
    "HH-RLHF latest checkpoint": (
        "/kaggle/working/alignment-assignment-check/"
        "outputs/dpo_hh_full/last_checkpoint"
    ),
    "HH-RLHF backup ZIP": (
        "/kaggle/working/hh_rlhf_dpo_backup.zip"
    ),
}

for name, path in paths.items():
    print(f"\n{name}: {'EXISTS' if os.path.exists(path) else 'MISSING'}")
    if os.path.isdir(path):
        print("  Files:", os.listdir(path))
    elif os.path.isfile(path):
        print("  Size (GB):", round(os.path.getsize(path) / 1e9, 2))


Original SFT checkpoint: EXISTS
  Files: ['config.json', 'model.safetensors', 'generation_config.json']

HH-RLHF best checkpoint: EXISTS
  Files: ['config.json', 'metrics.json', 'generation_config.json', 'model.safetensors']

HH-RLHF latest checkpoint: EXISTS
  Files: ['config.json', 'optimizer.pt', 'metrics.json', 'trainer_state.pt', 'generation_config.json', 'model.safetensors']

HH-RLHF backup ZIP: EXISTS
  Size (GB): 1.83


In [33]:
import os
import torch

# Original SFT checkpoint — both models start from here
SFT_CHECKPOINT = (
    "/kaggle/input/datasets/khadijawn/"
    "stf-backup/outputs/sft_run/best_checkpoint"
)

# New output directory — keeps HH-RLHF results untouched
OUTPUT_DIR = (
    "/kaggle/working/alignment-assignment-check/"
    "outputs/dpo_ultrafeedback_full"
)
LAST_DIR = os.path.join(OUTPUT_DIR, "last_checkpoint")
BEST_DIR = os.path.join(OUTPUT_DIR, "best_checkpoint")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify that this is a fresh output directory
assert os.path.isdir(SFT_CHECKPOINT)
assert os.path.isfile(os.path.join(SFT_CHECKPOINT, "model.safetensors"))
assert not os.path.exists(LAST_DIR), (
    "A last_checkpoint already exists. Stop and inspect it before continuing."
)
assert not os.path.exists(BEST_DIR), (
    "A best_checkpoint already exists. Stop and inspect it before continuing."
)

print("SFT checkpoint:", SFT_CHECKPOINT)
print("UltraFeedback output:", OUTPUT_DIR)
print("HH-RLHF output remains untouched.")
print("GPU count:", torch.cuda.device_count())
print("CUDA available:", torch.cuda.is_available())

SFT checkpoint: /kaggle/input/datasets/khadijawn/stf-backup/outputs/sft_run/best_checkpoint
UltraFeedback output: /kaggle/working/alignment-assignment-check/outputs/dpo_ultrafeedback_full
HH-RLHF output remains untouched.
GPU count: 2
CUDA available: True


In [35]:
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/working/alignment-assignment-check/data/dpo"
)

ultra_train = load_dpo_json("ultrafeedback_train.json")
ultra_validation = load_dpo_json("ultrafeedback_validation.json")

print("UltraFeedback training pairs:", len(ultra_train))
print("UltraFeedback validation pairs:", len(ultra_validation))

assert len(ultra_train) == 15000
assert len(ultra_validation) == 1000

print("\nSample keys:", ultra_train[0].keys())

UltraFeedback training pairs: 15000
UltraFeedback validation pairs: 1000

Sample keys: dict_keys(['prompt', 'chosen', 'rejected'])


In [36]:
import inspect

print("train_dpo_resumable:")
print(inspect.signature(train_dpo_resumable))

print("\nevaluate_dpo:")
print(inspect.signature(evaluate_dpo))

print("\nModel-loading functions in this notebook:")
for name, obj in list(globals().items()):
    if callable(obj) and any(
        word in name.lower()
        for word in ["load_model", "checkpoint", "build_model"]
    ):
        try:
            print(f"\n{name}{inspect.signature(obj)}")
        except (TypeError, ValueError):
            print(name)

train_dpo_resumable:
(train_rows, validation_rows, resume=False, seed=42)

evaluate_dpo:
(policy_model, reference_model, rows, label)

Model-loading functions in this notebook:

save_latest_checkpoint(model, optimizer, state)

load_latest_checkpoint(model, optimizer)

save_best_checkpoint(model, metrics)


In [37]:
import inspect

print(inspect.getsource(train_dpo_resumable))

def train_dpo_resumable(
    train_rows,
    validation_rows,
    resume=False,
    seed=42,
):
    """
    Train one DPO epoch.

    Resume only from a checkpoint made by this function.
    If resume=False, the supplied policy must be the original SFT model.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    optimizer = torch.optim.RMSprop(
        policy.parameters(),
        lr=LEARNING_RATE,
    )

    if resume:
        state = load_latest_checkpoint(policy, optimizer)
        indices = state["indices"]
    else:
        if LAST_DIR.exists():
            raise FileExistsError(
                f"{LAST_DIR} already exists. Use resume=True to continue, "
                "or choose a new OUTPUT_DIR for a fresh experiment."
            )

        indices = list(range(len(train_rows)))
        random.Random(seed).shuffle(indices)

        state = {
            "epoch": 0,
            "global_step": 0,
            "next_index": 0,
        

In [38]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM

# Original SFT checkpoint
SFT_CHECKPOINT = Path(
    "/kaggle/input/datasets/khadijawn/"
    "stf-backup/outputs/sft_run/best_checkpoint"
)

# Fresh UltraFeedback output directory
OUTPUT_DIR = Path(
    "/kaggle/working/alignment-assignment-check/"
    "outputs/dpo_ultrafeedback_full"
)
LAST_DIR = OUTPUT_DIR / "last_checkpoint"
BEST_DIR = OUTPUT_DIR / "best_checkpoint"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Never overwrite an existing run
assert not LAST_DIR.exists(), f"Checkpoint already exists: {LAST_DIR}"
assert not BEST_DIR.exists(), f"Checkpoint already exists: {BEST_DIR}"

# Load the trainable policy in FP32 on GPU 0
print("Loading policy from the original SFT checkpoint...")
policy = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    torch_dtype=torch.float32,
    local_files_only=True,
).to("cuda:0")

# Load the frozen reference in FP16 on GPU 1
print("Loading reference from the original SFT checkpoint...")
reference = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    torch_dtype=torch.float16,
    local_files_only=True,
).to("cuda:1")

# Freeze the reference model
reference.eval()
for parameter in reference.parameters():
    parameter.requires_grad_(False)

# Confirm the setup
print("\nPolicy device:", next(policy.parameters()).device)
print("Policy dtype:", next(policy.parameters()).dtype)
print("Reference device:", next(reference.parameters()).device)
print("Reference dtype:", next(reference.parameters()).dtype)
print(
    "Reference trainable parameters:",
    sum(p.numel() for p in reference.parameters() if p.requires_grad)
)
print("Output directory:", OUTPUT_DIR)
print("Model setup complete.")

Loading policy from the original SFT checkpoint...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading reference from the original SFT checkpoint...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Policy device: cuda:0
Policy dtype: torch.float32
Reference device: cuda:1
Reference dtype: torch.float16
Reference trainable parameters: 0
Output directory: /kaggle/working/alignment-assignment-check/outputs/dpo_ultrafeedback_full
Model setup complete.


In [40]:
import gc
import torch

gc.collect()

for gpu in range(torch.cuda.device_count()):
    with torch.cuda.device(gpu):
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info(gpu)
        print(
            f"GPU {gpu}: "
            f"{free / 1024**3:.2f} GiB free / "
            f"{total / 1024**3:.2f} GiB total"
        )

GPU 0: 1.13 GiB free / 14.56 GiB total
GPU 1: 13.38 GiB free / 14.56 GiB total


In [41]:
import inspect

print("MAX_LENGTH =", MAX_LENGTH)
print("Gradient checkpointing =", policy.is_gradient_checkpointing)

print("\n--- calculate_pair ---")
print(inspect.getsource(calculate_pair))

print("\n--- chunked_response_logps ---")
print(inspect.getsource(chunked_response_logps))

print("\n--- response_batch ---")
print(inspect.getsource(response_batch))

MAX_LENGTH = 512
Gradient checkpointing = False

--- calculate_pair ---
def calculate_pair(policy_model, reference_model, row):
    """Compute one pair's DPO loss and reward margin."""
    chosen_batch = response_batch(row["prompt"], row["chosen"])
    rejected_batch = response_batch(row["prompt"], row["rejected"])

    policy_chosen = chunked_response_logps(policy_model, chosen_batch)
    policy_rejected = chunked_response_logps(policy_model, rejected_batch)

    with torch.no_grad():
        reference_chosen = chunked_response_logps(
            reference_model, chosen_batch
        )
        reference_rejected = chunked_response_logps(
            reference_model, rejected_batch
        )

    # The two models live on different GPUs, so move only the scalar
    # reference results to the policy GPU.
    device = policy_chosen.device
    reference_chosen = reference_chosen.to(device)
    reference_rejected = reference_rejected.to(device)

    margin = BETA * (
        (policy_chosen 

In [42]:
import gc
import torch

# Clear any partial gradients left by the failed backward pass
if "policy" in globals():
    for parameter in policy.parameters():
        parameter.grad = None

gc.collect()

for gpu in range(torch.cuda.device_count()):
    with torch.cuda.device(gpu):
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info(gpu)
        print(
            f"GPU {gpu}: {free / 1024**3:.2f} GiB free "
            f"of {total / 1024**3:.2f} GiB"
        )

GPU 0: 1.66 GiB free of 14.56 GiB
GPU 1: 13.38 GiB free of 14.56 GiB


In [43]:
# Keep the trainable policy in FP32
policy.config.use_cache = False

# Enable memory-saving activation recomputation
policy.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

print("Policy gradient checkpointing:", policy.is_gradient_checkpointing)
print("Policy dtype:", next(policy.parameters()).dtype)
print("Policy device:", next(policy.parameters()).device)
print("Reference device:", next(reference.parameters()).device)

Policy gradient checkpointing: True
Policy dtype: torch.float32
Policy device: cuda:0
Reference device: cuda:1


In [45]:
import gc
import torch

# Reduce sequence length for the UltraFeedback run
MAX_LENGTH = 256

# Keep the memory-saving setting enabled
policy.config.use_cache = False

if not policy.is_gradient_checkpointing:
    policy.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

# Clear gradients left by the failed backward pass
for parameter in policy.parameters():
    parameter.grad = None

gc.collect()

for gpu in range(torch.cuda.device_count()):
    with torch.cuda.device(gpu):
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info(gpu)
        print(
            f"GPU {gpu}: {free / 1024**3:.2f} GiB free "
            f"/ {total / 1024**3:.2f} GiB total"
        )

print("MAX_LENGTH:", MAX_LENGTH)
print("Gradient checkpointing:", policy.is_gradient_checkpointing)

GPU 0: 5.22 GiB free / 14.56 GiB total
GPU 1: 13.38 GiB free / 14.56 GiB total
MAX_LENGTH: 256
Gradient checkpointing: True


In [ ]:
ultra_metrics = train_dpo_resumable(
    train_rows=ultra_train,
    validation_rows=ultra_validation,
    resume=True,
    seed=42,
)

print("\nFinal UltraFeedback DPO metrics:")
print(ultra_metrics)

Loading model weights from: /kaggle/working/alignment-assignment-check/outputs/dpo_ultrafeedback_full/last_checkpoint
Restored missing lm_head.weight from shared embeddings.
Model weights restored successfully.
Optimizer state restored.
Resuming at global step 0, training index 0.
Step 1 | Pairs 32/15000 | Train loss=0.6918 | Grad norm=59.65
Step 2 | Pairs 64/15000 | Train loss=0.6849 | Grad norm=61.50
Step 3 | Pairs 96/15000 | Train loss=0.6812 | Grad norm=66.21
Step 4 | Pairs 128/15000 | Train loss=0.6087 | Grad norm=60.85
Step 5 | Pairs 160/15000 | Train loss=0.6950 | Grad norm=67.17
Step 6 | Pairs 192/15000 | Train loss=0.6150 | Grad norm=51.59
Step 7 | Pairs 224/15000 | Train loss=0.6585 | Grad norm=50.58
Step 8 | Pairs 256/15000 | Train loss=0.6546 | Grad norm=57.24
Step 9 | Pairs 288/15000 | Train loss=0.6655 | Grad norm=63.71
Step 10 | Pairs 320/15000 | Train loss=0.6471 | Grad norm=53.65


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved recovery checkpoint at optimizer step 10.
Step 11 | Pairs 352/15000 | Train loss=0.6126 | Grad norm=62.25
Step 12 | Pairs 384/15000 | Train loss=0.6401 | Grad norm=59.95
Step 13 | Pairs 416/15000 | Train loss=0.6083 | Grad norm=68.64
Step 14 | Pairs 448/15000 | Train loss=0.5408 | Grad norm=58.37
Step 15 | Pairs 480/15000 | Train loss=0.7448 | Grad norm=77.07
Step 16 | Pairs 512/15000 | Train loss=0.5862 | Grad norm=53.34
Step 17 | Pairs 544/15000 | Train loss=0.6873 | Grad norm=72.01
Step 18 | Pairs 576/15000 | Train loss=0.7118 | Grad norm=73.50
Step 19 | Pairs 608/15000 | Train loss=0.6037 | Grad norm=57.16
Step 20 | Pairs 640/15000 | Train loss=0.6437 | Grad norm=58.65


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved recovery checkpoint at optimizer step 20.
Step 21 | Pairs 672/15000 | Train loss=0.6803 | Grad norm=63.57
Step 22 | Pairs 704/15000 | Train loss=0.6748 | Grad norm=60.69
Step 23 | Pairs 736/15000 | Train loss=0.5824 | Grad norm=56.54
Step 24 | Pairs 768/15000 | Train loss=0.6160 | Grad norm=57.99
Step 25 | Pairs 800/15000 | Train loss=0.5829 | Grad norm=54.97
Step 26 | Pairs 832/15000 | Train loss=0.6070 | Grad norm=58.59
Step 27 | Pairs 864/15000 | Train loss=0.6917 | Grad norm=58.99
Step 28 | Pairs 896/15000 | Train loss=0.5278 | Grad norm=59.03
Step 29 | Pairs 928/15000 | Train loss=0.6222 | Grad norm=49.63
Step 30 | Pairs 960/15000 | Train loss=0.7879 | Grad norm=73.55


...

Saved recovery checkpoint at optimizer step 270.
Step 271 | Pairs 8672/15000 | Train loss=0.5432 | Grad norm=50.80
Step 272 | Pairs 8704/15000 | Train loss=0.6823 | Grad norm=79.50
Step 273 | Pairs 8736/15000 | Train loss=0.6719 | Grad norm=64.74
Step 274 | Pairs 8768/15000 | Train loss=0.6786 | Grad norm=68.45
Step 275 | Pairs 8800/15000 | Train loss=0.7355 | Grad norm=67.45
Step 276 | Pairs 8832/15000 | Train loss=0.6299 | Grad norm=64.83
Step 277 | Pairs 8864/15000 | Train loss=0.5804 | Grad norm=50.93
Step 278 | Pairs 8896/15000 | Train loss=0.6765 | Grad norm=65.34
Step 279 | Pairs 8928/15000 | Train loss=0.4733 | Grad norm=51.04
Step 280 | Pairs 8960/15000 | Train loss=0.5696 | Grad norm=65.03


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved recovery checkpoint at optimizer step 280.
Step 281 | Pairs 8992/15000 | Train loss=0.5555 | Grad norm=54.97
Step 282 | Pairs 9024/15000 | Train loss=0.5679 | Grad norm=57.18
Step 283 | Pairs 9056/15000 | Train loss=0.6707 | Grad norm=58.20
Step 284 | Pairs 9088/15000 | Train loss=0.6541 | Grad norm=64.99
Step 285 | Pairs 9120/15000 | Train loss=0.6859 | Grad norm=61.75
Step 286 | Pairs 9152/15000 | Train loss=0.6243 | Grad norm=67.51
Step 287 | Pairs 9184/15000 | Train loss=0.6338 | Grad norm=66.37
Step 288 | Pairs 9216/15000 | Train loss=0.7372 | Grad norm=67.90
Step 289 | Pairs 9248/15000 | Train loss=0.6097 | Grad norm=61.90
Step 290 | Pairs 9280/15000 | Train loss=0.6194 | Grad norm=65.92


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved recovery checkpoint at optimizer step 290.
Step 291 | Pairs 9312/15000 | Train loss=0.6367 | Grad norm=53.97
Step 292 | Pairs 9344/15000 | Train loss=0.7301 | Grad norm=119.01
Step 293 | Pairs 9376/15000 | Train loss=0.6966 | Grad norm=66.36
Step 294 | Pairs 9408/15000 | Train loss=0.5735 | Grad norm=57.66
Step 295 | Pairs 9440/15000 | Train loss=0.6160 | Grad norm=55.73
Step 296 | Pairs 9472/15000 | Train loss=0.6028 | Grad norm=58.35


In [8]:

from pathlib import Path

folder = Path("/kaggle/input/datasets/khadijawn/aa-step8")

print("Folder exists:", folder.exists())

if folder.exists():
    files = [p for p in folder.rglob("*") if p.is_file()]
    print("Total files:", len(files))

    for f in files:
        print(f.relative_to(folder), f"({f.stat().st_size / (1024**2):.1f} MB)")


Folder exists: True
Total files: 18
submission_adapter.py (0.0 MB)
.gitignore (0.0 MB)
pyproject.toml (0.0 MB)
README.md (0.0 MB)
requirements.txt (0.0 MB)
tests/toy_lm.py (0.0 MB)
tests/test_public_adapter_smoke.py (0.0 MB)
tests/__init__.py (0.0 MB)
alignment_pre_sft_backup/submission_adapter.py (0.0 MB)
docs/assignment_manual.pdf (0.2 MB)
eval_sets/hh_rlhf_eval_300_seed42.json (0.1 MB)
data/persona_lora/README.md (0.0 MB)
data/persona_lora/ramsay_style_preferences.json (0.1 MB)
alignment_pre_sft_backup/eval_sets/alpaca_eval_300_seed42_baseline.json (0.3 MB)
alignment_pre_sft_backup/eval_sets/alpaca_eval_300_seed42_baseline_settings.json (0.0 MB)
alignment_pre_sft_backup/eval_sets/alpaca_eval_300_seed42_prompts.json (0.1 MB)
alignment_pre_sft_backup/data/sft/validation.json (2.6 MB)
alignment_pre_sft_backup/data/sft/train.json (78.0 MB)


In [1]:
from pathlib import Path

repo = Path("/kaggle/working/alignment-assignment-check")

print("Repository exists:", repo.exists())

if repo.exists():
    for name in ["outputs", ".git", ".gitignore"]:
        p = repo / name
        print(name, "exists:", p.exists())
        

Repository exists: False
